# DACON 236754 — 최종 End-to-End 파이프라인 (Colab A100 기준)

공식 데이터 + `unlabeled.csv`(2만 행 라벨) + `human_labels.csv`를 받아 아래를 한 번에 수행합니다.

**데이터 검증 → 법령/RAG 자산 생성 → `unlabeled.csv` 자동 라벨(19,600행) weak 사례 → 사람 확정 라벨(400행) 사례 → Gemma dev 추론/평가 → 선택적 pseudo 사례 → 최종 dev 비교 → 공개 sample test → 제출 형식 검사 → `submit.zip`**

**GPU 프로필 (VRAM으로 자동 선택)**

| | `a100` (Colab A100 40/80GB, 기본) | `rtx3090` (24GB 서버) |
|---|---|---|
| 모델 | `google/gemma-4-26B-A4B-it` 원본(대회 고정 revision) | `cyankiwi/gemma-4-26B-A4B-it-AWQ-4bit` |
| 양자화 | `int8_per_channel_weight_only` (**평가 서버와 동일**) | `awq_marlin` (4bit) |
| 컨텍스트 | 16384 (평가 서버와 동일) | 12288 |
| 동시 시퀀스 | 4 (80GB면 8) | 2 |

A100 프로필은 평가 서버와 같은 모델·양자화·컨텍스트라서 dev 점수를 제출 성능의 추정치로 볼 수 있습니다.

**Colab 사용 팁**
- 결과·체크포인트는 Google Drive(`DRIVE_DIR`)에, venv·모델 캐시는 빠른 로컬 디스크(`/content`)에 둡니다.
- 런타임이 끊기면 1번 셀의 `RESUME_WORK`에 이전 작업 폴더를 넣고 전체를 다시 실행하세요. 끝난 dev 추론은 재사용되고, 끊긴 추론은 체크포인트(`*.partial.jsonl`)에서 이어갑니다(모델은 다시 받습니다).
- Gemma 모델은 HF에서 라이선스 동의가 필요합니다. Colab 보안 비밀(Secrets)에 `HF_TOKEN`을 등록하세요.

**RAG 방법론** (자세한 설명·실험 결과는 `docs/추론.md`)
- 구조 기반 청킹(조항 제목 단위 + 상위 제목 접두), Contextual Retrieval(규칙 CTX / 선택: Gemma 생성)
- 하이브리드 검색: BGE-M3 dense + 문자 bigram BM25, 가중 RRF → 항목별 위반 1 + 비위반 근접 1 대조 사례
- 사례 항목 9개(법령 발췌 항목과 분리), 사례는 근거 중심 420자

**대회 제약**
- 고정 LLM `google/gemma-4-26B-A4B-it`, 학습·LoRA 없음, 제출 ZIP에 가중치 없음
- `output/submission.csv` 49열(`id, v1~v24, e1~e24`), v는 정수 0/1, e는 원문 부분문자열 500자 이하, 부재탐지 e10/e11/e16/e18/e20은 빈칸
- dev 점수를 잴 때는 dev 정답 사례를 RAG에서 제외(누수 방지). dev gold 사례는 단계 선택이 끝난 뒤 제출 자산에만 추가


## 1. 경로 · GPU 프로필 · 실험 설정


In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, datetime, zipfile, gzip, csv

IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()

# =========================
# 저장 위치
# =========================
USE_DRIVE = True                          # Colab: 결과·체크포인트를 Drive에 저장(런타임 끊김 대비)
DRIVE_DIR = '/content/drive/MyDrive/pps_dacon'
RESUME_WORK = ''                          # 끊긴 실행을 이어가려면 이전 WORK 경로(예: .../pps_dacon/pps_final_20260928_101500)

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_BASE = Path(DRIVE_DIR)
elif IN_COLAB:
    WORK_BASE = Path('/content/pps_runs')
else:
    WORK_BASE = Path(os.environ.get('PPS_WORK_BASE', Path.cwd().parent/'pps_runs'))
FAST_DIR = Path('/content/pps_local') if IN_COLAB else WORK_BASE   # venv·HF 캐시(용량 큼)는 로컬 디스크
FAST_DIR.mkdir(parents=True, exist_ok=True)
WORK = Path(RESUME_WORK) if RESUME_WORK else WORK_BASE/('pps_final_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
WORK.mkdir(parents=True, exist_ok=True)
print('IN_COLAB =', IN_COLAB, '| WORK =', WORK, '| FAST_DIR =', FAST_DIR)

# =========================
# 입력 데이터 — 비워두면 Colab 업로드 창에서 받음(open.zip, unlabeled.csv, human_labels.csv)
# =========================
OPEN_DIR = ''            # open 폴더 경로(예: Drive의 /content/drive/MyDrive/pps_dacon/open) — 지정하면 업로드 생략
OPEN_ZIP_PATH = ''       # open.zip 경로
UNLABELED_CSV_PATH = ''  # 비우면 OPEN_DIR/unlabeled.csv 또는 업로드 파일
HUMAN_LABELS_CSV_PATH = ''
HUMAN_ROWS = 400         # human_labels.csv가 없을 때 앞 N행을 사람 라벨로 간주
AUDIT_ZIP_PATH = ''      # (선택) 이전 labeling_audit_v2_bundle.zip

# =========================
# GPU 프로필 — 'auto'면 VRAM으로 선택(≥38GB → a100)
# =========================
GPU_PROFILE = 'auto'
try:
    _q = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'], text=True).strip().splitlines()[0]
    GPU_NAME, VRAM_MB = _q.split(',')[0].strip(), int(_q.split(',')[1])
except Exception as e:
    GPU_NAME, VRAM_MB = 'unknown', 0
    print('nvidia-smi 확인 실패:', e)
if GPU_PROFILE == 'auto':
    GPU_PROFILE = 'a100' if VRAM_MB >= 38000 else 'rtx3090'
PROFILES = {
    # 평가 서버와 동일: 원본 모델 + int8 weight-only + 16K 컨텍스트
    'a100':    dict(model_id='google/gemma-4-26B-A4B-it', revision='4d7ae4984b7db7de8f8457170b3f1a419ee76d52',
                    quant='int8_per_channel_weight_only', max_len=16384, gpu_mem=0.88,
                    seqs=8 if VRAM_MB >= 79000 else 4, batch=8 if VRAM_MB >= 79000 else 4, batched=8192 if VRAM_MB >= 79000 else 4096),
    # 24GB: 원본·int8이 안 들어가 AWQ 4bit 커뮤니티 체크포인트
    'rtx3090': dict(model_id='cyankiwi/gemma-4-26B-A4B-it-AWQ-4bit', revision=None,
                    quant='awq_marlin', max_len=12288, gpu_mem=0.92, seqs=2, batch=4, batched=4096),
}
_p = PROFILES[GPU_PROFILE]
LOCAL_MODEL_ID, LOCAL_MODEL_REVISION = _p['model_id'], _p['revision']
LOCAL_QUANTIZATION, LOCAL_MAX_MODEL_LEN, LOCAL_GPU_MEM = _p['quant'], _p['max_len'], _p['gpu_mem']
LOCAL_MAX_NUM_SEQS, LOCAL_BATCH_SIZE, LOCAL_MAX_BATCHED_TOKENS = _p['seqs'], _p['batch'], _p['batched']
LOCAL_KV_CACHE_DTYPE = 'auto'   # 메모리 부족하면 'fp8'
LOCAL_ENFORCE_EAGER = False
LOCAL_MODEL_DIR = os.environ.get('PPS_LOCAL_MODEL_DIR', '')   # 이미 받아둔 모델 폴더가 있으면 지정
LOCAL_EMBED_DIR = os.environ.get('PPS_LOCAL_EMBED_DIR', '')
HF_CACHE = FAST_DIR/'hf_cache'
print(f'GPU: {GPU_NAME} ({VRAM_MB/1024:.0f} GiB) → 프로필 {GPU_PROFILE}: {LOCAL_MODEL_ID} · {LOCAL_QUANTIZATION} · ctx {LOCAL_MAX_MODEL_LEN} · seqs {LOCAL_MAX_NUM_SEQS}')
print('디스크 여유 GiB (FAST_DIR):', round(shutil.disk_usage(FAST_DIR).free/2**30, 1), '— a100 프로필은 원본 모델 약 52GB 다운로드')

# =========================
# 파이프라인 스위치
# =========================
DEV_LIMIT = None             # 빠른 점검: 예) 20 → dev 앞 20건만 추론/채점
RUN_PSEUDO = True
ADD_DEV_GOLD_TO_SUBMIT = True
PSEUDO_PER_ITEM = 25
PSEUDO_KEEP_PER_ITEM = 15

# unlabeled.csv 사용 방식
USE_AUTO_LABELS = True       # 자동 라벨 행(19,600) → weak 사례
AUTO_POS_CAP_PER_ITEM = 40   # 항목별 자동 라벨 양성 사례 상한(규칙 점수 높은 순)
AUTO_NEG_PER_ITEM = 4        # 항목별 자동 라벨 hard negative 수(자동 0은 누락 위험이 있어 적게)
USE_HUMAN_LABELS = True      # 사람 확정 행(400) → human 사례
HUMAN_NEG_PER_ITEM = 8
EVAL_HUMAN_HOLDOUT = True    # 사람 라벨 레코드 자체로도 채점(같은 공고 사례는 검색에서 자동 제외)
HOLDOUT_LIMIT = 150          # 홀드아웃 채점 건수(400건 전부는 3090에서 오래 걸림). None이면 전부
USE_AUDIT_BUNDLE = False     # 이전 감사 번들 weak 후보를 추가로 쓸지
AUDIT_CASE_CAP_PER_ITEM = 80

# =========================
# 실제 제출 서버와 동일하게 사용할 추론 기본값
# =========================
PROD_MAX_MODEL_LEN = 16384
PROD_MAX_TOKENS = 1536
PROD_RETRY_TOKENS = 2048
PROD_BATCH_SIZE = 4
PROD_MAX_NUM_SEQS = 4
PROD_MAX_BATCHED_TOKENS = 4096
PROD_MAX_CASE_LABELS = 9   # 사례를 붙일 항목 수(4→9: dev 정답 위반 항목 커버리지 40.5%→59.5%)
PROD_GPU_MEM = 0.88
PROD_SEED = 42
PROD_QUANTIZATION = 'int8_per_channel_weight_only'

# =========================
# RAG 검색(하이브리드) — 아래 값이 script.py 기본값으로 기록되어 제출 서버에서도 그대로 동작
# =========================
RET_MODE = 'hybrid'        # 'hybrid'(dense+BM25 등 가중 RRF) | 'dense'(기존 방식) | 'lexical'(BGE 없이)
RET_MAX_LEN = 1024         # BGE-M3 입력 토큰 상한(기존 384)
RET_POS = 1                # 항목별 위반 사례 수
RET_NEG = 1                # 항목별 비위반 근접 사례 수
RET_W_DENSE, RET_W_SPARSE, RET_W_BM25, RET_W_META = 1.0, 0.0, 1.0, 0.0   # dev proxy: dense+BM25가 가장 안정적(learned sparse·meta는 손해 → 0)
COMPARE_RETRIEVAL = False  # True면 9-C에서 기존 dense 검색으로 dev를 한 번 더 돌려 A/B 비교(추론 1회 추가)

# Contextual Retrieval (Anthropic): 사례/질의 조각 앞에 '어느 공고의 어느 조항인지' 문맥을 붙여 임베딩·BM25
CONTEXTUAL = True          # 규칙 기반 문맥(meta + 조항 제목) — 비용 없음, 기본 켬
CONTEXTUAL_LLM = False     # True면 7-A에서 Gemma가 사례별 문맥 1~2문장을 생성해 자산에 저장(GPU 추가 시간)
CTX_MAX_DOC_CHARS = 5000   # Gemma 문맥 생성 시 넣을 공고 원문 길이
os.environ['PPS_CONTEXTUAL'] = '1' if CONTEXTUAL else '0'

# 청킹: 공고문 조항 구조(3. / 가. / ①) 기준 + 상위 조항 제목 접두 (근거 문구 포함률: 문서 발췌 3500자 0.60→0.71, 1700자 0.29→0.49)
CHUNKING = 'structure'     # 'structure' | 'fixed'(이전 900/1100자 고정)
os.environ['PPS_CHUNKING'] = CHUNKING

V1_BASELINE_MACRO_F1 = 0.113768


## 2. Python 환경 (대회 고정 버전: vllm 0.26.0 · torch 2.11.0 · transformers 5.14.1 · xgrammar 0.2.3)

- 별도 Python 3.12 venv를 `FAST_DIR/venv312`에 만듭니다(Colab은 세션마다 새로 설치, 서버는 재사용).
- 이미 위 버전이 설치된 환경이면 `USE_CURRENT_PYTHON = True`.


In [ ]:
USE_CURRENT_PYTHON = False

if USE_CURRENT_PYTHON:
    PYTHON = Path(sys.executable)
else:
    VENV = FAST_DIR/'venv312'
    PYTHON = VENV/'bin'/'python'
    if not PYTHON.exists():
        if IN_COLAB:
            subprocess.run(['apt-get', 'install', '-y', '-qq', 'ninja-build'], check=False)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
        subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', '--seed', str(VENV)], check=True)
        subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(PYTHON),
            'vllm==0.26.0', 'torch==2.11.0', 'transformers==5.14.1', 'xgrammar==0.2.3',
            'huggingface-hub', 'ninja', 'numpy'], check=True)
    else:
        print('기존 venv 재사용:', VENV)

# 노트북 커널 쪽 의존성(채점 표 출력용)
try:
    import pandas as pd
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pandas'], check=True)
    import pandas as pd

check = '''
import shutil, torch
from importlib.metadata import version
print('ninja:', shutil.which('ninja'))
for x in ['vllm','torch','transformers','xgrammar']:
    print(x, version(x))
print('CUDA:', torch.version.cuda, '| available:', torch.cuda.is_available())
p = torch.cuda.get_device_properties(0)
print('GPU:', p.name, '| VRAM GiB:', round(p.total_memory/2**30, 1), '| sm', f'{p.major}{p.minor}')
'''
subprocess.run([str(PYTHON), '-c', check], check=True, env={**os.environ, 'PATH': f"{Path(PYTHON).parent}:{os.environ.get('PATH','')}"})


## 3. 데이터 준비 — 공식 데이터 + `unlabeled.csv` + `human_labels.csv`

- `OPEN_DIR`(Drive 폴더 등)나 `OPEN_ZIP_PATH`를 지정하지 않으면 Colab 업로드 창이 열립니다. **`open.zip`, `unlabeled.csv`, `human_labels.csv`**를 한 번에 선택하세요(파일 이름·내용으로 자동 구분).
- `unlabeled.csv`: 2만 행 전체(49열). 앞 400행은 사람 확정 라벨, 나머지는 규칙 기반 자동 라벨
- 빈칸(판단 불가)은 어떤 단계에서도 사례·채점에 쓰지 않습니다.


In [ ]:
HEADER49 = ['id'] + [f'v{i}' for i in range(1,25)] + [f'e{i}' for i in range(1,25)]

def count_jsonl_gz(p):
    with gzip.open(p, 'rt', encoding='utf-8') as f:
        return sum(1 for x in f if x.strip())

# ---- 입력 파일 찾기: 경로 지정 → 없으면 Colab 업로드
def _is_open_zip(p):
    with zipfile.ZipFile(p) as z:
        n = z.namelist()
    return any(x.endswith('train_unlabeled.jsonl.gz') for x in n) and any(x.endswith('dev_labels.csv') for x in n)
def _is_audit_zip(p):
    with zipfile.ZipFile(p) as z:
        return any(x.endswith('labels_with_review_mask.jsonl.gz') for x in z.namelist())
def _csv_rows(p):
    with open(p, encoding='utf-8-sig', newline='') as f:
        rd = csv.reader(f); h = next(rd, [])
        return h, sum(1 for r in rd if r)

if not OPEN_DIR and not OPEN_ZIP_PATH and IN_COLAB:
    from google.colab import files
    UPLOAD = WORK/'upload'; UPLOAD.mkdir(exist_ok=True)
    print('open.zip, unlabeled.csv, human_labels.csv 를 선택하세요')
    for name, blob in files.upload().items():
        p = UPLOAD/name; p.write_bytes(blob)
        if p.suffix.lower() == '.zip':
            if _is_open_zip(p): OPEN_ZIP_PATH = str(p)
            elif _is_audit_zip(p): AUDIT_ZIP_PATH = str(p)
        elif p.suffix.lower() == '.csv':
            h, n = _csv_rows(p)
            if h == HEADER49 and n >= 20000 and not UNLABELED_CSV_PATH: UNLABELED_CSV_PATH = str(p)
            elif h == HEADER49 and 0 < n < 20000 and not HUMAN_LABELS_CSV_PATH: HUMAN_LABELS_CSV_PATH = str(p)
elif not OPEN_DIR and not OPEN_ZIP_PATH:
    OPEN_DIR = str(Path.cwd())

if OPEN_ZIP_PATH:
    OPEN_EXTRACT = WORK/'open_extract'
    if not OPEN_EXTRACT.exists():
        with zipfile.ZipFile(OPEN_ZIP_PATH) as z:
            z.extractall(OPEN_EXTRACT)
    roots = [p.parent for p in OPEN_EXTRACT.rglob('train_unlabeled.jsonl.gz')]
    assert len(roots) == 1, roots
    ROOT = roots[0]
else:
    ROOT = Path(OPEN_DIR)
UNLABELED_CSV = Path(UNLABELED_CSV_PATH) if UNLABELED_CSV_PATH else ROOT/'unlabeled.csv'
HUMAN_LABELS_CSV = Path(HUMAN_LABELS_CSV_PATH) if HUMAN_LABELS_CSV_PATH else ROOT/'human_labels.csv'
required = [ROOT/'README.md', ROOT/'dev.jsonl.gz', ROOT/'dev_labels.csv', ROOT/'train_unlabeled.jsonl.gz',
            ROOT/'data'/'test.jsonl.gz', ROOT/'data'/'항목표.json', ROOT/'data'/'정답스키마_디코딩.json',
            ROOT/'data'/'법령패키지'/'중기부고시'/'중기부고시_경쟁제품_세부품명.csv']
for p in required:
    assert p.exists(), f'필수 파일 없음: {p}'
print('ROOT =', ROOT)
print('dev:', count_jsonl_gz(ROOT/'dev.jsonl.gz'), '| train_unlabeled:', count_jsonl_gz(ROOT/'train_unlabeled.jsonl.gz'),
      '| public sample test:', count_jsonl_gz(ROOT/'data'/'test.jsonl.gz'))
assert count_jsonl_gz(ROOT/'train_unlabeled.jsonl.gz') == 20000

# ---- unlabeled.csv 검증
assert UNLABELED_CSV.exists(), f'unlabeled.csv 없음: {UNLABELED_CSV}'
with open(UNLABELED_CSV, encoding='utf-8-sig', newline='') as f:
    rd = csv.reader(f); hdr = next(rd); ul_rows = [r for r in rd if r]
assert hdr == HEADER49, 'unlabeled.csv 헤더가 49열 형식이 아닙니다'
with gzip.open(ROOT/'train_unlabeled.jsonl.gz', 'rt', encoding='utf-8') as f:
    train_ids = [json.loads(l)['id'] for l in f if l.strip()]
assert [r[0] for r in ul_rows] == train_ids, 'unlabeled.csv id/순서가 train_unlabeled와 다릅니다'
bad = [(r[0], k) for r in ul_rows for k, x in enumerate(r[1:25], 1) if x.strip() not in ('0', '1', '')]
assert not bad, f'0/1/빈칸 외 값: {bad[:5]}'
cells = [x.strip() for r in ul_rows for x in r[1:25]]
print(f"unlabeled.csv: {len(ul_rows)}행 · 양성 {cells.count('1')} · 0 {cells.count('0')} · 빈칸 {cells.count('')}")

# ---- 사람/자동 구분
HUMAN_IDS_ARG = str(HUMAN_LABELS_CSV) if HUMAN_LABELS_CSV.exists() else ''
if HUMAN_IDS_ARG:
    with open(HUMAN_LABELS_CSV, encoding='utf-8-sig', newline='') as f:
        human_ids = {r['id'] for r in csv.DictReader(f) if any((r[f'v{i}'] or '').strip() in ('0','1') for i in range(1,25))}
    print('사람 라벨 id:', len(human_ids), '(human_labels.csv 기준)')
else:
    human_ids = {r[0] for r in ul_rows[:HUMAN_ROWS]}
    print('사람 라벨 id:', len(human_ids), f'(앞 {HUMAN_ROWS}행 기준)')
LABEL_SELECT_ARGS = ['--labels', str(UNLABELED_CSV), '--human-ids', HUMAN_IDS_ARG, '--human-rows', str(HUMAN_ROWS)]
print('자동 라벨 행:', sum(1 for r in ul_rows if r[0] not in human_ids and any(x.strip() in ('0','1') for x in r[1:25])))

# ---- (선택) 이전 감사 번들
AUDIT_LABELS = None
if USE_AUDIT_BUNDLE and AUDIT_ZIP_PATH:
    AUDIT_ROOT = WORK/'audit_bundle'; AUDIT_ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(AUDIT_ZIP_PATH) as z:
        z.extractall(AUDIT_ROOT)
    AUDIT_LABELS = next(AUDIT_ROOT.rglob('labels_with_review_mask.jsonl.gz'))
    print('audit bundle:', AUDIT_LABELS)

# ---- dev 입력(빠른 점검용 DEV_LIMIT)
DEV_INPUT = ROOT/'dev.jsonl.gz'
if DEV_LIMIT:
    DEV_INPUT = WORK/f'dev_first{DEV_LIMIT}.jsonl.gz'
    with gzip.open(ROOT/'dev.jsonl.gz', 'rt', encoding='utf-8') as f, gzip.open(DEV_INPUT, 'wt', encoding='utf-8') as g:
        for k, line in enumerate(x for x in f if x.strip()):
            if k >= DEV_LIMIT: break
            g.write(line)
    print('DEV_LIMIT 적용:', DEV_INPUT)


## 4. V2 코드 생성 (`v2_core.py`, `build_assets.py`, `script.py`)

**RAG 검색 = 하이브리드(기본 dense + BM25, 가중 RRF 결합)**
- BGE-M3 **dense(CLS) 벡터** (learned sparse 가중치도 계산 가능하지만 dev 검색 proxy에서 손해라 기본 가중치 0)
- 문자 bigram **BM25**로 "소기업·소상공인", "인접", "확약서" 같은 법정 용어 일치를 보강
- (선택) **meta 유사도**(국가/지방, 물품/용역, 수의, 협상, 금액구간) 채널 — dev 검색 proxy에서 오히려 낮아 기본 가중치 0(`RET_W_META`로 켤 수 있음)
- 네 순위를 가중 **Reciprocal Rank Fusion**(k=60)으로 결합 → 항목별 **위반 사례 + 비위반 근접 사례**를 공고 중복 없이 선택
- Gemma 프롬프트에는 두 사례를 "대조 사례"로 나란히 제시해 위반/비위반이 갈리는 지점을 비교하게 함
- 사례를 붙이는 항목을 9개로 늘리고(법령 발췌 항목 수와 분리), 사례는 "항목+압축 meta+근거 중심 창"으로 짧게(420자) 표시
- BGE 로드 실패 시 BM25+meta로, sparse head가 없으면 dense+BM25+meta로 자동 축소
- **Contextual Retrieval**(Anthropic): 사례·질의 조각마다 `CTX` 줄(계약법·업무·계약방법·금액구간·발주기관 + 조항 제목 경로)을 붙여 임베딩과 BM25를 계산. `CONTEXTUAL_LLM=True`면 Gemma가 이 줄을 1~2문장 문맥으로 다시 써서 자산에 저장(7-A). 프롬프트에는 CTX 줄을 넣지 않음
- **구조 기반 청킹**: 공고문을 조항 제목(3. / 가. / ①)에서 나누고 같은 조항은 ~700자까지 합치며, 긴 조항은 15% 겹침 창으로 자르고, 조각마다 상위 조항 제목(`〔3. 입찰참가자격〕`)을 붙임. 검색 질의와 Gemma 문서 발췌 모두에 적용


In [ ]:
from pathlib import Path
CORE_SOURCE = 'from __future__ import annotations\nimport csv, gzip, hashlib, io, json, math, os, re, unicodedata\nfrom collections import Counter\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, Iterator, List, Optional, Tuple\n\nITEMS=[f"v{i}" for i in range(1,25)]\nABSENCE={"v10","v11","v16","v18","v20"}\nDOC_ORDER={"공고문":0,"규격서":1,"과업지시서":2,"제안요청서":3,"예외공표서":4,"기타":5}\nMETA_FIELDS=["적용계약법","업무구분","계약방법","낙찰방법","낙찰하한율","배정예산금액","입찰추정가격","소관구분","공동도급구성방식","정보화사업여부","세부품명번호목록","제한지역코드목록","지역제한여부","면허업종제한목록","업종제한여부","조항호내용","공고게시일자","개찰예정일자","긴급공고여부","입찰방법","조달방식"]\n\nSPECIAL_RULES={\n"v1":"참가자격을 특정 기관·단체 종류로 한정했는지 확인하고 법령상 허용 사유와 구분한다.",\n"v2":"고시금액 미만에서 실적제한을 두었는지 확인한다. 소액수의 등 적용 예외를 함께 본다.",\n"v3":"실적 요구 금액을 사업예산·추정가격과 비교해 1배수 이상인지 숫자로 확인한다.",\n"v4":"고시금액 이상에서 특정 기관 또는 특정 실적으로 참가자격을 제한했는지 확인한다.",\n"v5":"고시금액 이상 물품·용역에 지역제한을 두었는지 적용 법과 기관 유형을 함께 확인한다.",\n"v6":"고시금액 미만 지역제한이 시·군·구급으로 과도하게 좁혀졌는지 본다. [지역:*|단위=기초]는 기초자치단체급 정보다.",\n"v7":"고시금액 미만 지역제한에서 허용 범위를 벗어난 인접지역 확대·설정인지 확인한다.",\n"v8":"실적제한과 지역제한을 동시에 부과한 중복제한인지 확인한다.",\n"v9":"규격서·과업지시서 등에 특정 제조사·브랜드·모델명이 실제 요구조건으로 명시됐는지 본다. 단순 예시·참고·호환 설명과 구분한다.",\n"v10":"중소기업자간 경쟁제품 적용대상인지 먼저 확인한 뒤, 전체 문서에 요구되는 직접생산확인 관련 조건이 빠졌는지 본다.",\n"v11":"중소기업자간 경쟁제품 적용대상인지 먼저 확인한 뒤, 전체 문서에 요구되는 중소기업자간 경쟁·참가 제한이 빠졌는지 본다.",\n"v12":"중기간 경쟁제품이 아닌 일반제품에 직접생산확인 제한을 부당하게 둔 경우인지 본다.",\n"v13":"중기간 경쟁제품에서 소기업·소상공인으로만 제한해 중기업을 배제했는지 본다. 금액구간 자체는 핵심 성립요건이 아니며 소액수의 예외를 확인한다.",\n"v14":"일반물품·용역과 고시금액 구간을 확인한 뒤 중소기업 제한이 허용되는지 판단한다.",\n"v15":"일반제품 + 1억원 이상~고시금액 미만 구간에서 소기업·소상공인 제한이 적법한지 예외와 함께 판단한다.",\n"v16":"일반제품 + 1억원 이상~고시금액 미만 적용대상인데 필요한 중소기업 제한이 전체 문서에 없는지 본다. 법정 예외 명시 여부를 확인한다.",\n"v17":"1억원 미만 일반제품에서 중소기업 제한 범위가 적정한지 소기업·소상공인 기준 및 예외와 비교한다.",\n"v18":"1억원 미만 일반제품 적용대상인데 필요한 소기업·소상공인 제한이 전체 문서에 없는지 본다. 법정 예외 명시 여부를 확인한다.",\n"v19":"물품공급·기술지원 확약서를 입찰 단계에서 미리 제출하도록 요구했는지 본다. 계약 체결 후 제출과 구분한다.",\n"v20":"SW/정보화 사업인지 먼저 확인하고 사업금액 구간별 대기업 참여제한 하한제도 적용 여부·근거가 전체 문서에 빠졌는지 본다.",\n"v21":"공동수급체 구성원의 최소 지분율·출자비율·분담률 5%/10% 기준 위반 여부를 숫자로 확인한다.",\n"v22":"협상에 의한 계약에서 현장설명회 참석 자체를 입찰참가자격으로 제한했는지 본다.",\n"v23":"지방계약 + 협상에 의한 계약에서 현장설명회 공고기간·일정 기준을 확인한다. 국가계약 건은 성립하지 않는다.",\n"v24":"공고문과 나라장터 meta의 불일치를 예산·계약방법·지역제한·업종 네 축만 비교한다. 다른 meta 차이는 v24 근거로 쓰지 않는다.",\n}\n\nKEYWORDS={\n"v1":["참가자격","입찰참가","대학","산학협력단","협회","기관만","법인만","참여 가능"],\n"v2":["실적","납품실적","이행실적","실적증명","최근 3년","추정가격"],\n"v3":["실적","납품실적","이행실적","사업예산","배정예산","추정가격","이상 실적"],\n"v4":["실적","특정기관","공공기관","기관 실적","유사실적","고시금액"],\n"v5":["지역제한","소재지","본점","주된 영업소","관내","지역"],\n"v6":["지역제한","소재지","본점","주된 영업소","단위=기초","시군구"],\n"v7":["지역제한","인접","연접","인근","광역","단위=기초"],\n"v8":["실적","지역제한","소재지","납품실적","이행실적"],\n"v9":["제조사","제작사","모델명","모델","브랜드","품번","제품번호","model"],\n"v10":["직접생산","직접생산확인","경쟁제품","세부품명"],\n"v11":["중소기업자간 경쟁","중소기업","중기업","소기업","소상공인","경쟁제품"],\n"v12":["직접생산","직접생산확인","세부품명","일반제품"],\n"v13":["소기업","소상공인","중기업","중소기업","경쟁제품"],\n"v14":["중소기업","중기업","소기업","소상공인","일반물품"],\n"v15":["소기업","소상공인","중소기업","1억원","일반물품"],\n"v16":["중소기업","중기업","소기업","소상공인","예외","판로지원"],\n"v17":["중소기업","소기업","소상공인","1억원","일반물품"],\n"v18":["소기업","소상공인","중소기업","예외","판로지원"],\n"v19":["확약서","공급확약","물품공급","기술지원확약","제출"],\n"v20":["소프트웨어","SW사업","정보화사업","대기업 참여","참여제한","사업금액"],\n"v21":["공동수급","공동도급","공동이행","분담이행","지분율","출자비율","분담률","최소지분"],\n"v22":["현장설명","설명회","참석업체","참가자격","협상"],\n"v23":["현장설명","설명회","공고기간","제안서","협상","공고게시"],\n"v24":["예산","추정가격","계약방법","지역제한","업종제한","나라장터"],\n}\n\ndef nfc(s: Any) -> str:\n    return unicodedata.normalize("NFC", "" if s is None else str(s))\n\ndef norm(s: Any) -> str:\n    return re.sub(r"\\s+"," ",nfc(s)).strip()\n\ndef open_text(path: str|Path):\n    return gzip.open(path,"rt",encoding="utf-8") if str(path).endswith(".gz") else open(path,"r",encoding="utf-8")\n\ndef load_records(path: str|Path, limit: Optional[int]=None) -> List[Dict[str,Any]]:\n    out=[]\n    with open_text(path) as f:\n        for line in f:\n            if not line.strip(): continue\n            r=json.loads(line)\n            if not isinstance(r.get("id"),str) or not isinstance(r.get("docs"),list) or not isinstance(r.get("meta"),dict):\n                raise ValueError("invalid record")\n            for d in r["docs"]:\n                d["text"]=nfc(d.get("text",""))\n            out.append(r)\n            if limit and len(out)>=limit: break\n    return out\n\ndef full_text(rec: Dict[str,Any]) -> str:\n    return "\\n".join(nfc(d.get("text","")) for d in rec.get("docs",[]))\n\ndef meta_text(rec: Dict[str,Any]) -> str:\n    m=rec.get("meta",{})\n    return "\\n".join(f"- {k}: {m.get(k) if m.get(k) is not None else \'미기재\'}" for k in META_FIELDS if k in m)\n\ndef parse_product_codes(v: Any) -> List[str]:\n    if v is None: return []\n    if isinstance(v,list): s=" ".join(map(str,v))\n    else: s=str(v)\n    return sorted(set(re.findall(r"(?<!\\d)(\\d{10})(?!\\d)",s)))\n\ndef load_competitive_products(csv_path: str|Path) -> Dict[str,Dict[str,str]]:\n    out={}\n    with open(csv_path,"r",encoding="utf-8-sig",newline="") as f:\n        for row in csv.DictReader(f):\n            code=re.sub(r"\\D","",row.get("세부품명번호","") or "")\n            if len(code)==10: out[code]={k:(v or "") for k,v in row.items()}\n    return out\n\ndef amount(rec: Dict[str,Any]) -> Optional[int]:\n    m=rec.get("meta",{})\n    for k in ("입찰추정가격","배정예산금액"):\n        v=m.get(k)\n        if isinstance(v,(int,float)) and v>0: return int(v)\n        if isinstance(v,str):\n            ds=re.sub(r"\\D","",v)\n            if ds: return int(ds)\n    return None\n\ndef is_local(rec): return "지방" in str(rec.get("meta",{}).get("적용계약법", ""))\n\ndef is_negotiation(rec):\n    m=rec.get("meta",{})\n    return "협상" in (str(m.get("계약방법", ""))+" "+str(m.get("낙찰방법", "")))\n\ndef _split_fixed(rec: Dict[str,Any], size: int=1100, overlap: int=120) -> List[Dict[str,str]]:\n    chunks=[]\n    for d in sorted(rec.get("docs",[]), key=lambda x:(DOC_ORDER.get(x.get("type","기타"),9),x.get("doc_id",""))):\n        t=nfc(d.get("text",""))\n        # Prefer paragraph boundaries, then fall back to windows.\n        paras=[p.strip() for p in re.split(r"\\n\\s*\\n",t) if p.strip()]\n        if not paras: paras=[t]\n        buf=""\n        for p in paras:\n            if len(buf)+len(p)+2<=size:\n                buf=(buf+"\\n"+p).strip()\n            else:\n                if buf: chunks.append({"type":d.get("type","기타"),"doc_id":d.get("doc_id",""),"text":buf})\n                if len(p)<=size: buf=p\n                else:\n                    step=max(200,size-overlap)\n                    for s in range(0,len(p),step):\n                        piece=p[s:s+size]\n                        if piece: chunks.append({"type":d.get("type","기타"),"doc_id":d.get("doc_id",""),"text":piece})\n                    buf=""\n        if buf: chunks.append({"type":d.get("type","기타"),"doc_id":d.get("doc_id",""),"text":buf})\n    return chunks\n\nCHUNKING=os.environ.get("PPS_CHUNKING","structure")   # structure | fixed\n_TOPH=re.compile(r"^\\s*(?:\\d{1,2}\\s*\\.(?!\\d)|[IVXⅠ-Ⅹ]{1,4}\\s*[.)]|제\\s*\\d+\\s*조|[□■◆◇○●▶]\\s*\\S{2,20}$)")\n_SUBH=re.compile(r"^\\s*(?:[가-하]\\s*[.)]|\\d{1,2}\\s*\\)|[①-⑳]|\\(\\d{1,2}\\)|\\([가-하]\\)|[-·•※◦∘ㆍ○]\\s)")\n\ndef _split_structure(rec: Dict[str,Any], size: int=1100, overlap_ratio: float=0.15) -> List[Dict[str,str]]:\n    """Structure-aware + hierarchical chunking for 공고문: cut at numbered clause headings (3. / 가. / ① ...),\n    merge small clauses of the same section up to ~65% of size, window over-long clauses with 15% overlap,\n    and prefix each chunk with its parent section heading so it stays interpretable on its own."""\n    out=[]; target=int(size*0.65)\n    for d in sorted(rec.get("docs",[]), key=lambda x:(DOC_ORDER.get(x.get("type","기타"),9),x.get("doc_id",""))):\n        t=nfc(d.get("text","")); top=""; units=[]; cur=[]; cur_top=""\n        for line in t.split("\\n"):\n            s=line.strip()\n            if not s: continue\n            is_top=bool(_TOPH.match(s))\n            if (is_top or _SUBH.match(s)) and cur:\n                units.append((cur_top,"\\n".join(cur))); cur=[]\n            if is_top: top=s[:60]\n            if not cur: cur_top=top\n            cur.append(s)\n        if cur: units.append((cur_top,"\\n".join(cur)))\n        buf=""; buf_top=""\n        def emit(head,txt):\n            pre=f"〔{head}〕\\n" if head and not txt.startswith(head[:15]) else ""\n            out.append({"type":d.get("type","기타"),"doc_id":d.get("doc_id",""),"head":head,"text":pre+txt})\n        for tp,u in units:\n            if len(u)>size:\n                if buf: emit(buf_top,buf); buf=""\n                step=max(200,int(size*(1-overlap_ratio)))\n                for s0 in range(0,len(u),step): emit(tp,u[s0:s0+size])\n                continue\n            if buf and (tp!=buf_top or len(buf)+len(u)+1>target):\n                emit(buf_top,buf); buf=""\n            if not buf: buf_top=tp\n            buf=(buf+"\\n"+u) if buf else u\n        if buf: emit(buf_top,buf)\n    return out\n\ndef split_doc_chunks(rec: Dict[str,Any], size: int=1100, overlap: int=120) -> List[Dict[str,str]]:\n    return _split_structure(rec,size) if CHUNKING=="structure" else _split_fixed(rec,size,overlap)\n\ndef toks(s: str) -> set[str]:\n    return set(re.findall(r"[가-힣A-Za-z0-9%]+",str(s).lower()))\n\ndef lexical_score(text: str, words: Iterable[str]) -> float:\n    lo=text.lower(); score=0.0\n    for w in words:\n        w=str(w).lower().strip()\n        if not w: continue\n        if w in lo: score += 2.0 if len(w)>=4 else 1.0\n    return score\n\ndef v24_signals(rec: Dict[str,Any]) -> List[str]:\n    t=full_text(rec)\n    m=rec.get("meta",{})\n    sig=[]\n    # Strong explicit contradictions only; these are hints, never final labels.\n    if m.get("지역제한여부")=="Y" and re.search(r"지역\\s*제한\\s*(없음|없다|미적용)",t): sig.append("meta는 지역제한 Y인데 문서에 \'지역제한 없음\' 취지 표현이 있음")\n    if m.get("지역제한여부")=="N" and re.search(r"(지역제한|소재지).{0,40}(제한|한함|이어야)",t,re.S): sig.append("meta는 지역제한 N인데 문서에 지역 제한성 표현이 있음")\n    if m.get("업종제한여부")=="Y" and re.search(r"업종\\s*제한\\s*(없음|없다|미적용)",t): sig.append("meta는 업종제한 Y인데 문서에 업종제한 없음 취지 표현이 있음")\n    if m.get("업종제한여부")=="N" and re.search(r"업종.{0,30}(제한|등록|면허)",t,re.S): sig.append("meta는 업종제한 N인데 문서에 업종/면허 제한성 표현이 있음")\n    # Contract-method conflict terms.\n    cm=str(m.get("계약방법","") or "")+" "+str(m.get("낙찰방법","") or "")\n    methods=[("일반경쟁","일반경쟁"),("제한경쟁","제한경쟁"),("수의","수의계약"),("협상","협상에 의한 계약")]\n    meta_tags={tag for kw,tag in methods if kw in cm}\n    doc_tags={tag for kw,tag in methods if kw in t}\n    if meta_tags and doc_tags and meta_tags.isdisjoint(doc_tags): sig.append(f"계약방법 후보 불일치: meta={sorted(meta_tags)}, 문서={sorted(doc_tags)}")\n    # Budget: only flag if the exact budget is absent while a labeled budget line contains another large amount.\n    b=m.get("배정예산금액")\n    if isinstance(b,(int,float)) and b>0:\n        ds=str(int(b))\n        digits_only=re.sub(r"[^0-9]","",t)\n        if ds not in digits_only:\n            near=re.findall(r"(?:예산|사업비|배정금액|추정금액).{0,35}?([0-9][0-9,]{6,})\\s*원?",t,re.S)\n            vals={int(x.replace(",","")) for x in near if x.replace(",","").isdigit()}\n            if vals and int(b) not in vals: sig.append(f"meta 배정예산 {int(b):,}원이 문서 예산성 금액 후보 {sorted(vals)[:4]}와 다름")\n    return sig\n\ndef candidate_scores(rec: Dict[str,Any], competitive_codes: set[str]) -> Tuple[Dict[str,float],List[str]]:\n    t=full_text(rec)\n    tl=t.lower()\n    m=rec.get("meta",{})\n    a=amount(rec)\n    codes=parse_product_codes(m.get("세부품명번호목록"))\n    competitive=bool(set(codes)&competitive_codes)\n    s={v:0.0 for v in ITEMS}\n    hints=[]\n    for v in ITEMS: s[v]+=lexical_score(t,KEYWORDS[v])*.35\n    perf=bool(re.search(r"(납품|이행|용역|계약)?\\s*실적|실적증명",t))\n    region=(m.get("지역제한여부")=="Y") or bool(re.search(r"지역제한|주된\\s*영업소|본점\\s*소재|소재지를",t))\n    if perf:\n        for v in ("v2","v3","v4","v8"): s[v]+=2\n    if region:\n        for v in ("v5","v6","v7","v8"): s[v]+=2\n    if perf and region: s["v8"]+=3\n    # 2026 national goods/services WTO threshold from distributed law snapshot; candidate hint only.\n    threshold=230_000_000\n    if a:\n        if a<threshold:\n            s["v2"]+=1; s["v6"]+=1; s["v7"]+=1\n        else:\n            s["v4"]+=1; s["v5"]+=1; s["v14"]+=1\n        if 100_000_000<=a<threshold:\n            s["v15"]+=1; s["v16"]+=1\n        if a<100_000_000:\n            s["v17"]+=1; s["v18"]+=1\n    if re.search(r"\\[지역:[^\\]]*단위=기초",t): s["v6"]+=3\n    if re.search(r"인접|연접|인근",t): s["v7"]+=3\n    # model / brand clues\n    if re.search(r"(제조사|모델명|브랜드|품번|제품번호)\\s*[:：]",t,re.I): s["v9"]+=4\n    if re.search(r"\\b[A-Z]{2,}[A-Z0-9._/-]{2,}\\b",t): s["v9"]+=1\n    # competitive product applicability\n    if competitive:\n        s["v10"]+=5; s["v11"]+=5; s["v13"]+=3\n        hints.append("중기부고시 경쟁제품 세부품명과 meta 세부품명번호가 일치")\n    elif codes:\n        s["v12"]+=2; s["v14"]+=1; s["v15"]+=1; s["v16"]+=1; s["v17"]+=1; s["v18"]+=1\n        hints.append("meta에 세부품명번호가 있으나 경쟁제품 코드 일치는 확인되지 않음")\n    small=bool(re.search(r"중소기업|중기업|소기업|소상공인",t))\n    direct=bool(re.search(r"직접생산(확인)?",t))\n    if direct:\n        s["v10"]+=1; s["v12"]+=3\n    if small:\n        for v in ("v11","v13","v14","v15","v17"): s[v]+=2\n    if competitive and re.search(r"소기업|소상공인",t) and "중기업" not in t: s["v13"]+=3\n    if re.search(r"(물품공급|기술지원).{0,30}확약서|확약서.{0,30}(입찰|제출)",t,re.S): s["v19"]+=5\n    sw=(str(m.get("정보화사업여부","")).upper() in {"Y","예","YES"}) or bool(re.search(r"소프트웨어|정보화\\s*사업|SW\\s*사업",t,re.I))\n    if sw:\n        s["v20"]+=5; hints.append("SW/정보화 사업 신호 있음")\n    if re.search(r"공동수급|공동도급|공동이행|분담이행",t) or m.get("공동도급구성방식") not in (None,"","미입력"):\n        s["v21"]+=3\n        if re.search(r"(?:지분율|출자비율|분담률|최소지분).{0,30}(?:5\\s*%|10\\s*%)",t,re.S): s["v21"]+=4\n    if re.search(r"현장\\s*설명|현장설명회",t):\n        s["v22"]+=4; s["v23"]+=3\n    if is_negotiation(rec):\n        s["v22"]+=1; s["v23"]+=1\n    for x in v24_signals(rec):\n        s["v24"]+=3; hints.append("v24 힌트: "+x)\n    if re.search(r"(대학|산학협력단|협회|재단|특정기관).{0,45}(만|한함|참여 가능|참가 가능)",t,re.S): s["v1"]+=3\n    return s,hints\n\ndef select_priority_labels(scores: Dict[str,float], max_labels: int=9) -> List[str]:\n    ranked=sorted(ITEMS,key=lambda v:(scores.get(v,0.0),-int(v[1:])),reverse=True)\n    strong=[v for v in ranked if scores.get(v,0)>=1.5]\n    return strong[:max_labels]\n\ndef select_doc_context(rec: Dict[str,Any], items: Dict[str,Any], max_chars: int=9000) -> str:\n    chunks=split_doc_chunks(rec)\n    if not chunks: return ""\n    selected=[]; used=0; covered=set()\n    # keep the opening of the notice and one opening attachment if possible\n    for i,c in enumerate(chunks):\n        if c["type"]=="공고문" and len(selected)<1:\n            selected.append(i); used+=len(c["text"])+40\n        elif c["type"]!="공고문" and len(selected)<2:\n            selected.append(i); used+=len(c["text"])+40\n            break\n    pairs=[]\n    for v in ITEMS:\n        words=[items[v].get("항목명","")]+KEYWORDS[v]\n        for i,c in enumerate(chunks):\n            sc=lexical_score(c["text"],words)\n            if sc>0: pairs.append((sc,v,i))\n    pairs.sort(reverse=True)\n    for sc,v,i in pairs:\n        if used>=max_chars: break\n        if i in selected:\n            covered.add(v); continue\n        bonus=2 if v not in covered else 0\n        if sc+bonus<1: continue\n        need=len(chunks[i]["text"])+50\n        if used+need>max_chars: continue\n        selected.append(i); covered.add(v); used+=need\n    selected=sorted(set(selected))\n    out=[]\n    for i in selected:\n        c=chunks[i]\n        out.append(f"[{c[\'type\']}:{c[\'doc_id\']}]\\n{c[\'text\']}")\n    dropped=rec.get("dropped_doc_counts") or {}\n    if dropped: out.append("[주의: 입력에서 제외된 문서] "+", ".join(f"{k} {v}건" for k,v in dropped.items()))\n    return "\\n\\n".join(out)\n\nCONTEXTUAL=os.environ.get("PPS_CONTEXTUAL","1")=="1"   # Contextual Retrieval: situate each case/query chunk in its notice\n_HEAD=re.compile(r"^\\s*(?:\\d{1,2}\\s*[.)]|[가-하]\\s*[.)]|[①-⑳]|[IVXⅠ-Ⅹ]{1,4}\\s*[.)])\\s*\\S")\n\ndef section_path(t: str, idx: int) -> str:\n    """Nearest numbered heading (e.g. \'3. 입찰참가자격\') and sub-heading (\'나. ...\') above position idx."""\n    top=sub=""\n    for line in reversed(t[:max(0,idx)].split("\\n")[-400:]):\n        s=line.strip()\n        if not s or not _HEAD.match(s): continue\n        if re.match(r"^\\d{1,2}\\s*[.)]|^[IVXⅠ-Ⅹ]{1,4}\\s*[.)]",s):\n            top=s[:40]; break\n        if not sub: sub=s[:40]\n    return " > ".join(x for x in (top,sub) if x)\n\ndef _amount_band(rec):\n    a=amount(rec)\n    if not a: return "금액 미기재"\n    return "1억 미만" if a<100_000_000 else "1억~고시금액 미만" if a<230_000_000 else "고시금액~5억 미만" if a<500_000_000 else "5억 이상"\n\ndef context_line(rec: Dict[str,Any], doc_type: str, section: str) -> str:\n    m=rec.get("meta",{})\n    kind="수의계약" if "수의" in str(m.get("계약방법","")) else str(m.get("계약방법","") or "")\n    return (f"CTX {m.get(\'적용계약법\',\'\')} {m.get(\'업무구분\',\'\')} {kind}({m.get(\'낙찰방법\',\'\')}) 공고, 추정가격 {_amount_band(rec)}, "\n            f"{m.get(\'소관구분\',\'\')} 발주 — [{doc_type}] \'{section or \'본문\'}\' 부분").replace("None","")\n\ndef _locate(rec, piece):\n    for d in rec.get("docs",[]):\n        t=nfc(d.get("text","")); i=t.find(piece)\n        if i>=0: return d.get("type","기타"),t,i\n    return None,None,-1\n\ndef best_label_snippet(rec: Dict[str,Any], label: str, items: Dict[str,Any], max_chars: int=1800, with_ctx: bool=False):\n    chunks=split_doc_chunks(rec,900,80)\n    words=[items[label].get("항목명","")]+KEYWORDS[label]\n    ranked=sorted(chunks,key=lambda c:lexical_score(c["text"],words),reverse=True)\n    picked=[]; used=0\n    for c in ranked[:4]:\n        if used>=max_chars: break\n        txt=c["text"][:max_chars-used]\n        picked.append(f"[{c[\'type\']}] {txt}"); used+=len(txt)\n    out="\\n".join(picked)\n    if not with_ctx: return out\n    ctx=""\n    if ranked:\n        first=next((l.strip() for l in ranked[0]["text"].split("\\n") if l.strip() and not l.startswith("〔")),"")[:80]\n        dt,t,i=_locate(rec,first) if first else (None,None,-1)\n        if t is not None: ctx=context_line(rec,dt,section_path(t,i))\n    return out,ctx\n\ndef evidence_context(rec: Dict[str,Any], ev: str, radius: Optional[int]=None, before: int=150, after: int=450, with_ctx: bool=False):\n    # Evidence-centred window: short lead-in so the evidence survives the prompt cut.\n    if not ev: return ("","") if with_ctx else ""\n    if radius is not None: before=after=radius\n    ev=nfc(ev).strip()\n    for d in rec.get("docs",[]):\n        t=nfc(d.get("text","")); idx=t.find(ev)\n        if idx>=0:\n            w=f"[{d.get(\'type\',\'기타\')}] "+t[max(0,idx-before):min(len(t),idx+len(ev)+after)]\n            return (w,context_line(rec,d.get("type","기타"),section_path(t,idx))) if with_ctx else w\n    return ("","") if with_ctx else ""\n\ndef case_text(rec: Dict[str,Any], label: str, items: Dict[str,Any], evidence: str="") -> str:\n    core,ctx=evidence_context(rec,evidence,with_ctx=True) if evidence else best_label_snippet(rec,label,items,1500,with_ctx=True)\n    m=rec.get("meta",{})\n    mini=" | ".join(f"{k}={m.get(k)}" for k in ("적용계약법","업무구분","계약방법","낙찰방법","배정예산금액","입찰추정가격","정보화사업여부","세부품명번호목록","지역제한여부","업종제한여부") if k in m)\n    head=f"{label} {items[label].get(\'항목명\',\'\')}\\nMETA {mini}\\n"\n    if CONTEXTUAL and ctx: head+=ctx+"\\n"\n    return f"{head}{core}"[:3200]\n\ndef retrieval_query(rec: Dict[str,Any], label: str, items: Dict[str,Any]) -> str:\n    return case_text(rec,label,items,"")\n\ndef clean_evidence(ev: Any, rec: Dict[str,Any], max_len: int=500) -> str:\n    if not ev: return ""\n    e=nfc(ev).replace("\\r","").strip()[:max_len]\n    if not e or e.startswith(("=","+","@")): return ""\n    return e if e in full_text(rec) else ""\n\ndef safe_overrides(row: Dict[str,Any], rec: Dict[str,Any]) -> Dict[str,Any]:\n    # Only near-deterministic applicability rules from item notes.\n    if not is_negotiation(rec):\n        row["v22"]=0; row["e22"]=""\n        row["v23"]=0; row["e23"]=""\n    elif not is_local(rec):\n        row["v23"]=0; row["e23"]=""\n    return row\n\ndef item_guide(items: Dict[str,Any]) -> str:\n    lines=[]\n    for v in ITEMS:\n        x=items[v]\n        refs=f"국가:{x.get(\'국가계약법\',\'\')} / 지방:{x.get(\'지방계약법\',\'\')}"\n        lines.append(f"{v} {x.get(\'항목명\',\'\')} | {SPECIAL_RULES[v]} | 참고조문 {refs}")\n    return "\\n".join(lines)\n\ndef case_display(text: str, n: int) -> str:\n    """Prompt view of a case: label line + compact meta + evidence-centred body, capped at n chars."""\n    lines=text.split("\\n"); head=lines[0] if lines else ""\n    meta=next((l for l in lines[1:3] if l.startswith("META ")),"")\n    body="\\n".join(l for l in lines[1:] if l!=meta and not l.startswith("CTX "))\n    m={}\n    for kv in meta[5:].split(" | "):\n        if "=" in kv: k,v=kv.split("=",1); m[k.strip()]=v.strip()\n    amt=m.get("입찰추정가격") or m.get("배정예산금액") or "?"\n    short=" / ".join(x for x in (m.get("적용계약법",""),m.get("업무구분",""),m.get("계약방법",""),m.get("낙찰방법",""),f"추정가격 {amt}") if x)\n    return (f"{head} ({short})\\n{body}")[:n]\n\ndef build_user_prompt(rec: Dict[str,Any], asset: Dict[str,Any], examples: Dict[str,List[Dict[str,Any]]], doc_chars: int=9000, max_law_labels: int=8, max_case_labels: int=7, case_chars: int=900) -> Tuple[str,Dict[str,Any]]:\n    items=asset["items"]; comp=set(asset.get("competitive_codes",[]))\n    scores,hints=candidate_scores(rec,comp)\n    priority=select_priority_labels(scores,max(max_law_labels,max_case_labels))\n    law_branch="지방" if is_local(rec) else "국가"\n    law=[]\n    for v in priority[:max_law_labels]:\n        refs=asset.get("law_refs",{}).get(v,{}).get(law_branch,[])[:2]\n        if refs:\n            law.append(f"[{v} 관련 법령]\\n"+"\\n---\\n".join(r["text"][:1800] for r in refs))\n    ex=[]\n    for v in priority[:max_case_labels]:\n        arr=examples.get(v,[])\n        if not arr: continue\n        bits=[]\n        for c in arr[:4]:\n            tag="위반 사례(정답=1)" if int(c[\'y\'])==1 else "비위반 근접 사례(정답=0)"\n            bits.append(f"- {tag}: {case_display(c[\'text\'],case_chars)}")\n        ex.append(f"[{v} 대조 사례 — 위반/비위반이 어디서 갈리는지 비교]\\n"+"\\n".join(bits))\n    sig=[f"{v}: 후보점수={scores[v]:.1f}" for v in priority]\n    sig+=hints\n    if rec.get("input_completeness",{}).get("완전관측") is False:\n        sig.append("입력이 완전관측이 아님: 검색 조각에 없다는 이유만으로 부재를 단정하지 말 것")\n    docs=select_doc_context(rec,items,doc_chars)\n    user=f"""[나라장터 meta]\\n{meta_text(rec)}\\n\\n[프로그램 사전검사 힌트 — 정답이 아니라 주의 신호]\\n"""+ ("\\n".join(sig) if sig else "강한 후보 신호 없음") + f"""\\n\\n[고정 매핑 법령 발췌]\\n{chr(10).join(law) if law else \'우선순위 항목에 추가 법령 발췌 없음\'}\\n\\n[유사 사례]\\n{chr(10).join(ex) if ex else \'유사 사례 없음\'}\\n\\n[공고 문서 발췌]\\n{docs}\\n\\n24개 항목을 각각 독립적으로 전부 판정하라. 후보점수가 낮아도 문서상 근거가 있으면 1로 판정할 수 있다. 반대로 힌트만으로 1을 확정하지 마라. 부재탐지(v10,v11,v16,v18,v20)는 \'적용대상인가\'와 \'전체 문서에 필수 고지가 실제로 없는가\'를 순서대로 확인하고 근거문구는 빈 문자열로 둔다. v24는 예산·계약방법·지역제한·업종 네 축만 본다. e 배열의 근거는 제공 문서에서 그대로 복사 가능한 원문 부분문자열만 쓰며 확실하지 않으면 빈 문자열로 둔다."""\n    return user,{"priority":priority,"scores":scores}\n\ndef system_prompt(items: Dict[str,Any]) -> str:\n    return """당신은 나라장터 입찰공고 법령 위반 검토자다. 서로 다른 24개 항목을 하나도 생략하지 않고 독립적으로 판정한다. 한 항목을 다른 항목의 포괄 라벨처럼 사용하지 않는다. 제공된 항목 정의·고정 법령·공고·meta·사례만 사용한다. 사례는 참고 패턴이며 법령보다 우선하지 않는다. 검색 발췌에 문구가 없다는 사실만으로 부재를 확정하지 않는다. 여러 위반은 동시에 1일 수 있다. 출력은 v 배열 24개와 e 배열 24개뿐이다. v[i]는 v{i+1}의 0/1, e[i]는 같은 항목 근거 원문이다.\\n\\n[24개 판정 기준]\\n"""+item_guide(items)\n'
BUILD_SOURCE = 'from __future__ import annotations\nimport argparse, csv, gzip, hashlib, heapq, json, os, re\nfrom pathlib import Path\nfrom typing import Any, Dict, List\nfrom v2_core import *\n\nALIASES={\n"국가계약법 시행령":"국가를 당사자로 하는 계약에 관한 법률 시행령",\n"국가계약법 시행규칙":"국가를 당사자로 하는 계약에 관한 법률 시행규칙",\n"국가계약법":"국가를 당사자로 하는 계약에 관한 법률",\n"지방계약법 시행령":"지방자치단체를 당사자로 하는 계약에 관한 법률 시행령",\n"지방계약법 시행규칙":"지방자치단체를 당사자로 하는 계약에 관한 법률 시행규칙",\n"지방계약법":"지방자치단체를 당사자로 하는 계약에 관한 법률",\n"정부 입찰·계약 집행기준":"정부 입찰·계약 집행기준",\n"정부입찰계약집행기준":"정부 입찰·계약 집행기준",\n"지방자치단체 입찰 및 계약 집행기준":"지방자치단체 입찰 및 계약 집행기준",\n"지방자치단체 입찰 및 계약집행기준":"지방자치단체 입찰 및 계약 집행기준",\n"공동계약운용요령":"공동계약운용요령",\n"중소기업제품 구매촉진 및 판로지원에 관한 법률":"중소기업제품 구매촉진 및 판로지원에 관한 법률",\n"소프트웨어진흥법":"소프트웨어 진흥법",\n"중소 소프트웨어사업자의 사업 참여 지원에 관한 지침":"중소 소프트웨어사업자의 사업 참여 지원에 관한 지침",\n}\n\ndef law_chunks(law_dir: Path) -> List[Dict[str,str]]:\n    out=[]\n    for p in sorted(law_dir.glob(\'*.txt\')):\n        text=p.read_text(encoding=\'utf-8-sig\',errors=\'ignore\')\n        # if article headings are present, split on them; otherwise fixed windows for 예규/지침.\n        starts=[m.start() for m in re.finditer(r\'(?m)^\\s*제\\s*\\d+\\s*조(?:의\\d+)?(?:\\([^\\n]*\\))?\',text)]\n        if len(starts)>=5:\n            starts=starts+[len(text)]\n            pieces=[text[starts[i]:starts[i+1]].strip() for i in range(len(starts)-1)]\n        else:\n            size,step=2600,2300\n            pieces=[text[i:i+size].strip() for i in range(0,len(text),step)]\n        for j,seg in enumerate(pieces):\n            if len(seg)<80: continue\n            out.append({\'source\':p.stem,\'chunk\':j,\'text\':seg[:3600]})\n    return out\n\ndef ref_score(ref: str, chunk: Dict[str,str]) -> float:\n    q=toks(ref); t=toks(chunk[\'source\']+\' \'+chunk[\'text\'][:1200])\n    sc=sum(1 for x in q if x in t)\n    for short,longname in ALIASES.items():\n        if short in ref and longname in chunk[\'source\']: sc+=20\n    for art in re.findall(r\'제\\s*\\d+\\s*조(?:의\\d+)?\',ref):\n        a=re.sub(r\'\\s+\',\'\',art)\n        if a in re.sub(r\'\\s+\',\'\',chunk[\'text\'][:1000]): sc+=15\n    for chap in re.findall(r\'제\\s*\\d+\\s*(?:장|절)\',ref):\n        if re.sub(r\'\\s+\',\'\',chap) in re.sub(r\'\\s+\',\'\',chunk[\'text\']): sc+=7\n    return float(sc)\n\ndef build_law_refs(items: Dict[str,Any], chunks: List[Dict[str,str]]) -> Dict[str,Any]:\n    out={}\n    for v,x in items.items():\n        out[v]={}\n        for branch,key in [(\'국가\',\'국가계약법\'),(\'지방\',\'지방계약법\')]:\n            ref=str(x.get(key,\'\') or \'\')\n            if not ref or \'해당 없음\' in ref:\n                out[v][branch]=[]; continue\n            ranked=sorted(((ref_score(ref,c),c) for c in chunks), key=lambda z:z[0], reverse=True)\n            sel=[]; seen=set()\n            for sc,c in ranked:\n                if sc<=0: break\n                if c[\'source\'] in seen and len(sel)>=2: continue\n                sel.append({\'source\':c[\'source\'],\'score\':sc,\'text\':c[\'text\']})\n                seen.add(c[\'source\'])\n                if len(sel)>=4: break\n            out[v][branch]=sel\n    return out\n\ndef read_labels(path: Path) -> Dict[str,Dict[str,str]]:\n    with path.open(\'r\',encoding=\'utf-8-sig\',newline=\'\') as f:\n        return {r[\'id\']:r for r in csv.DictReader(f)}\n\ndef base_cases(dev_records, labels, items, competitive_codes):\n    by_id={r[\'id\']:r for r in dev_records}\n    cases=[]\n    for v in ITEMS:\n        # all gold positives\n        for rid,row in labels.items():\n            if int(row[v])==1 and rid in by_id:\n                ev=row.get(\'e\'+v[1:],\'\') or \'\'\n                cases.append({\'source_id\':rid,\'label\':v,\'y\':1,\'source\':\'gold\',\'text\':case_text(by_id[rid],v,items,ev)})\n        # hard negatives: candidate-score-high negatives\n        neg=[]\n        for rid,row in labels.items():\n            if int(row[v])!=0 or rid not in by_id: continue\n            scores,_=candidate_scores(by_id[rid],competitive_codes)\n            neg.append((scores[v],rid))\n        neg.sort(reverse=True)\n        for sc,rid in neg[:8]:\n            cases.append({\'source_id\':rid,\'label\':v,\'y\':0,\'source\':\'gold\',\'text\':case_text(by_id[rid],v,items,\'\')})\n    return cases\n\ndef build_base(root: Path, out: Path):\n    data=root/\'data\'\n    raw=json.loads((data/\'항목표.json\').read_text(encoding=\'utf-8-sig\'))\n    items=raw[\'항목\']\n    products=load_competitive_products(data/\'법령패키지\'/\'중기부고시\'/\'중기부고시_경쟁제품_세부품명.csv\')\n    comp=set(products)\n    laws=law_chunks(data/\'법령패키지\'/\'법령\')\n    refs=build_law_refs(items,laws)\n    dev=load_records(root/\'dev.jsonl.gz\')\n    labels=read_labels(root/\'dev_labels.csv\')\n    cases=base_cases(dev,labels,items,comp)\n    asset={\n        \'version\':\'v2-rule-rag-cases-1\',\n        \'items\':items,\n        \'absence\':sorted(ABSENCE),\n        \'competitive_codes\':sorted(comp),\n        \'competitive_product_meta\':{k:{\'세부품명\':v.get(\'세부품명\',\'\'),\'특이사항\':v.get(\'특이사항\',\'\')} for k,v in products.items()},\n        \'law_refs\':refs,\n        \'cases\':cases,\n        \'build_manifest\':{\'gold_dev_records\':len(dev),\'gold_cases\':len(cases),\'pseudo_cases\':0},\n    }\n    out.parent.mkdir(parents=True,exist_ok=True)\n    out.write_text(json.dumps(asset,ensure_ascii=False),encoding=\'utf-8\')\n    print(json.dumps(asset[\'build_manifest\'],ensure_ascii=False,indent=2))\n    print(\'asset MB\',round(out.stat().st_size/2**20,2))\n\ndef mine(root: Path, asset_path: Path, out_jsonl: Path, out_map: Path, per_item: int):\n    asset=json.loads(asset_path.read_text(encoding=\'utf-8\'))\n    comp=set(asset[\'competitive_codes\'])\n    heaps={v:[] for v in ITEMS}\n    train=root/\'train_unlabeled.jsonl.gz\'\n    with gzip.open(train,\'rt\',encoding=\'utf-8\') as f:\n        for line in f:\n            if not line.strip(): continue\n            r=json.loads(line)\n            scores,_=candidate_scores(r,comp)\n            rid=r[\'id\']\n            tie=int(hashlib.md5(rid.encode()).hexdigest()[:8],16)/2**32\n            for v in ITEMS:\n                sc=float(scores[v])\n                if sc<1.5: continue\n                item=(sc,tie,rid)\n                h=heaps[v]\n                if len(h)<per_item: heapq.heappush(h,item)\n                elif item>h[0]: heapq.heapreplace(h,item)\n    selected={}\n    for v,h in heaps.items():\n        for sc,tie,rid in sorted(h,reverse=True):\n            selected.setdefault(rid,{})[v]=round(sc,3)\n    out_map.write_text(json.dumps(selected,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    ids=set(selected)\n    out_jsonl.parent.mkdir(parents=True,exist_ok=True)\n    with gzip.open(train,\'rt\',encoding=\'utf-8\') as src, gzip.open(out_jsonl,\'wt\',encoding=\'utf-8\') as dst:\n        n=0\n        for line in src:\n            if not line.strip(): continue\n            r=json.loads(line)\n            if r[\'id\'] in ids:\n                r[\'_candidate_labels\']=selected[r[\'id\']]\n                dst.write(json.dumps(r,ensure_ascii=False)+\'\\n\'); n+=1\n    print(\'pseudo candidate records\',n,\'per-item cap\',per_item)\n    print(\'label candidate counts\',{v:sum(v in x for x in selected.values()) for v in ITEMS})\n\ndef augment(root: Path, asset_path: Path, pred_path: Path, map_path: Path, cand_path: Path, out: Path, max_per_item: int):\n    asset=json.loads(asset_path.read_text(encoding=\'utf-8\'))\n    cmap=json.loads(map_path.read_text(encoding=\'utf-8\'))\n    preds={}\n    with pred_path.open(\'r\',encoding=\'utf-8\',newline=\'\') as f:\n        for row in csv.DictReader(f): preds[row[\'id\']]=row\n    recs={r[\'id\']:r for r in load_records(cand_path)}\n    items=asset[\'items\']; add=[]\n    for v in ITEMS:\n        cands=[]\n        for rid,vs in cmap.items():\n            if v not in vs or rid not in preds or rid not in recs: continue\n            row=preds[rid]\n            if str(row[v])!=\'1\': continue\n            ev=row.get(\'e\'+v[1:],\'\') or \'\'\n            if v not in ABSENCE and not ev: continue\n            cands.append((float(vs[v]),rid,ev))\n        cands.sort(reverse=True)\n        for sc,rid,ev in cands[:max_per_item]:\n            add.append({\'source_id\':rid,\'label\':v,\'y\':1,\'source\':\'pseudo\',\'score\':sc,\'text\':case_text(recs[rid],v,items,ev)})\n    # de-duplicate label/id and append\n    seen={(c[\'source_id\'],c[\'label\'],c[\'y\']) for c in asset[\'cases\']}\n    add=[c for c in add if (c[\'source_id\'],c[\'label\'],c[\'y\']) not in seen]\n    asset[\'cases\'].extend(add)\n    asset[\'build_manifest\'][\'pseudo_cases\']=len(add)\n    asset[\'build_manifest\'][\'pseudo_source_records\']=len(set(c[\'source_id\'] for c in add))\n    out.write_text(json.dumps(asset,ensure_ascii=False),encoding=\'utf-8\')\n    print(json.dumps(asset[\'build_manifest\'],ensure_ascii=False,indent=2))\n\nif __name__==\'__main__\':\n    ap=argparse.ArgumentParser(); sp=ap.add_subparsers(dest=\'cmd\',required=True)\n    a=sp.add_parser(\'base\'); a.add_argument(\'--root\',required=True); a.add_argument(\'--out\',required=True)\n    a=sp.add_parser(\'mine\'); a.add_argument(\'--root\',required=True); a.add_argument(\'--asset\',required=True); a.add_argument(\'--out\',required=True); a.add_argument(\'--map\',required=True); a.add_argument(\'--per-item\',type=int,default=40)\n    a=sp.add_parser(\'augment\'); a.add_argument(\'--root\',required=True); a.add_argument(\'--asset\',required=True); a.add_argument(\'--pred\',required=True); a.add_argument(\'--map\',required=True); a.add_argument(\'--candidates\',required=True); a.add_argument(\'--out\',required=True); a.add_argument(\'--max-per-item\',type=int,default=30)\n    x=ap.parse_args()\n    if x.cmd==\'base\': build_base(Path(x.root),Path(x.out))\n    elif x.cmd==\'mine\': mine(Path(x.root),Path(x.asset),Path(x.out),Path(x.map),x.per_item)\n    else: augment(Path(x.root),Path(x.asset),Path(x.pred),Path(x.map),Path(x.candidates),Path(x.out),x.max_per_item)\n'
SCRIPT_SOURCE = 'from __future__ import annotations\nimport argparse, csv, gzip, io, json, os, sys, time, unicodedata\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Tuple\nimport numpy as np\nfrom v2_core import *\n\nHERE=Path(__file__).resolve().parent\nDEFAULT_ASSET=HERE/\'model\'/\'assets.json\'\nCOLS=[\'id\']+[f\'v{i}\' for i in range(1,25)]+[f\'e{i}\' for i in range(1,25)]\n\ndef log(x): print(\'[v2]\',x,file=sys.stderr,flush=True)\n\ndef schema():\n    return {\n        \'type\':\'object\',\'additionalProperties\':False,\'required\':[\'v\',\'e\'],\n        \'properties\':{\n            \'v\':{\'type\':\'array\',\'minItems\':24,\'maxItems\':24,\'items\':{\'type\':\'integer\',\'enum\':[0,1]}},\n            \'e\':{\'type\':\'array\',\'minItems\':24,\'maxItems\':24,\'items\':{\'type\':\'string\',\'maxLength\':500}},\n        }\n    }\n\ndef write_csv(rows,path):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    with path.open(\'w\',encoding=\'utf-8\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=COLS,lineterminator=\'\\n\'); w.writeheader(); w.writerows(rows)\n\ndef parse_output(obj,rec):\n    if not isinstance(obj,dict) or not isinstance(obj.get(\'v\'),list) or not isinstance(obj.get(\'e\'),list):\n        raise ValueError(\'structured output shape\')\n    if len(obj[\'v\'])!=24 or len(obj[\'e\'])!=24: raise ValueError(\'structured output length\')\n    row={\'id\':rec[\'id\']}; invalid=0\n    for i in range(24):\n        hit=1 if obj[\'v\'][i]==1 else 0\n        label=f\'v{i+1}\'; ev=\'\'\n        if hit and label not in ABSENCE:\n            ev=clean_evidence(obj[\'e\'][i],rec)\n            if obj[\'e\'][i] and not ev: invalid+=1\n        row[label]=hit; row[f\'e{i+1}\']=ev\n    return safe_overrides(row,rec),invalid\n\ndef mock_rows(records):\n    rows=[]\n    for r in records:\n        row={\'id\':r[\'id\']}\n        for i in range(1,25): row[f\'v{i}\']=0; row[f\'e{i}\']=\'\'\n        rows.append(row)\n    return rows\n\n# ---------------------------------------------------------------------------\n# Hybrid retrieval: BGE-M3 dense + BGE-M3 learned sparse + char-bigram BM25 + meta,\n# fused with weighted Reciprocal Rank Fusion, then contrastive pos/neg selection.\n# ---------------------------------------------------------------------------\nRET_MODE=os.environ.get(\'PPS_RET_MODE\',\'hybrid\')            # hybrid | dense | lexical\nRET_EMBED_MAX_LEN=int(os.environ.get(\'PPS_RET_MAX_LEN\',\'1024\'))\nRET_POS=int(os.environ.get(\'PPS_RET_POS\',\'1\'))               # violation examples per label\nRET_NEG=int(os.environ.get(\'PPS_RET_NEG\',\'1\'))               # near-miss non-violation examples per label\nRET_RRF_K=60\nRET_W={\'dense\':float(os.environ.get(\'PPS_RET_W_DENSE\',\'1.0\')),\n       \'sparse\':float(os.environ.get(\'PPS_RET_W_SPARSE\',\'0.0\')),   # dev proxy: BGE learned sparse는 오히려 낮음\n       \'bm25\':float(os.environ.get(\'PPS_RET_W_BM25\',\'1.0\')),\n       \'meta\':float(os.environ.get(\'PPS_RET_W_META\',\'0.0\'))}   # dev proxy: meta 채널은 기본 끔\n\ndef _device():\n    try:\n        import torch\n        return \'cuda\' if torch.cuda.is_available() else \'cpu\'\n    except Exception:\n        return \'cpu\'\n\ndef embed_texts(texts: List[str], embed_dir: str, batch_size: int=8, max_length: int=1024, want_sparse: bool=True):\n    """One BGE-M3 forward pass -> (L2-normalised CLS dense vectors, learned sparse token weights or None)."""\n    import torch\n    import torch.nn.functional as F\n    from transformers import AutoTokenizer, AutoModel\n    dev=_device()\n    tok=AutoTokenizer.from_pretrained(embed_dir,local_files_only=True,trust_remote_code=True)\n    model=AutoModel.from_pretrained(embed_dir,local_files_only=True,trust_remote_code=True,\n        torch_dtype=torch.bfloat16 if dev==\'cuda\' else torch.float32).to(dev).eval()\n    sparse_head=None\n    sp_path=Path(embed_dir)/\'sparse_linear.pt\'\n    if want_sparse and sp_path.exists():\n        try:\n            st=torch.load(sp_path,map_location=\'cpu\')\n            sparse_head=torch.nn.Linear(st[\'weight\'].shape[1],st[\'weight\'].shape[0])\n            sparse_head.load_state_dict(st); sparse_head=sparse_head.to(dev).float().eval()\n        except Exception as e:\n            log(f\'sparse head unavailable: {type(e).__name__}: {e}\'); sparse_head=None\n    skip={i for i in (tok.cls_token_id,tok.eos_token_id,tok.pad_token_id,tok.unk_token_id) if i is not None}\n    dense=[]; sparse=[] if sparse_head is not None else None\n    order=sorted(range(len(texts)),key=lambda i:len(texts[i]))          # length-bucketed batches\n    buf_d=[None]*len(texts); buf_s=[None]*len(texts)\n    with torch.inference_mode():\n        for s in range(0,len(order),batch_size):\n            idx=order[s:s+batch_size]\n            batch=tok([texts[i] for i in idx],padding=True,truncation=True,max_length=max_length,return_tensors=\'pt\').to(dev)\n            h=model(**batch).last_hidden_state\n            z=F.normalize(h[:,0].float(),p=2,dim=1).cpu().numpy()\n            if sparse_head is not None:\n                w=torch.relu(sparse_head(h.float())).squeeze(-1).cpu().numpy()\n                ids=batch[\'input_ids\'].cpu().numpy(); msk=batch[\'attention_mask\'].cpu().numpy()\n            for j,i in enumerate(idx):\n                buf_d[i]=z[j]\n                if sparse_head is not None:\n                    d={}\n                    for t,wt,m in zip(ids[j],w[j],msk[j]):\n                        if not m or int(t) in skip or wt<=0: continue\n                        t=int(t)\n                        if wt>d.get(t,0.0): d[t]=float(wt)\n                    buf_s[i]=d\n            if s%(batch_size*32)==0: log(f\'BGE {min(s+batch_size,len(texts))}/{len(texts)}\')\n    import gc\n    del model, tok, sparse_head\n    gc.collect()\n    if dev==\'cuda\':\n        torch.cuda.empty_cache()\n        try: torch.cuda.synchronize()\n        except Exception: pass\n    D=np.stack(buf_d) if buf_d else np.zeros((0,1),dtype=np.float32)\n    return D,(buf_s if sparse is not None else None)\n\ndef _sparse_dot(a,b):\n    if len(a)>len(b): a,b=b,a\n    return sum(w*b.get(t,0.0) for t,w in a.items())\n\ndef _bigrams(s: str):\n    s=re.sub(r\'\\s+\',\'\',nfc(s).lower())\n    return [s[i:i+2] for i in range(len(s)-1)]\n\nclass _BM25:\n    def __init__(self,docs,k1=1.2,b=0.75):\n        self.tf=[Counter(_bigrams(d)) for d in docs]\n        self.len=[sum(t.values()) for t in self.tf]; self.avg=(sum(self.len)/max(1,len(self.len))) or 1.0\n        df=Counter(); [df.update(t.keys()) for t in self.tf]\n        n=len(docs); self.idf={t:math.log(1+(n-c+0.5)/(c+0.5)) for t,c in df.items()}; self.k1=k1; self.b=b\n    def score(self,q,i):\n        tf=self.tf[i]; L=self.len[i]; s=0.0\n        for t in set(q):\n            f=tf.get(t)\n            if f: s+=self.idf[t]*f*(self.k1+1)/(f+self.k1*(1-self.b+self.b*L/self.avg))\n        return s\n\ndef _band(a):\n    if not a: return \'na\'\n    return \'<1억\' if a<100_000_000 else \'1억~고시\' if a<230_000_000 else \'고시~5억\' if a<500_000_000 else \'5억+\'\n\ndef _meta_key(m: Dict[str,Any]):\n    cm=str(m.get(\'계약방법\',\'\'))+\' \'+str(m.get(\'낙찰방법\',\'\'))\n    a=None\n    for k in (\'입찰추정가격\',\'배정예산금액\'):\n        ds=re.sub(r\'\\D\',\'\',str(m.get(k) or \'\'))\n        if ds and int(ds)>=1_000_000: a=int(ds); break\n    return (\'지방\' in str(m.get(\'적용계약법\',\'\')), \'물품\' in str(m.get(\'업무구분\',\'\')), \'수의\' in cm, \'협상\' in cm, _band(a))\n\ndef _case_meta(text: str):\n    m={}\n    line=next((l for l in text.split(\'\\n\') if l.startswith(\'META \')),\'\')\n    for kv in line[5:].split(\' | \'):\n        if \'=\' in kv: k,v=kv.split(\'=\',1); m[k.strip()]=v.strip()\n    return _meta_key(m)\n\ndef _meta_sim(a,b):\n    return sum(1.0 for x,y in zip(a,b) if x==y)/len(a)\n\ndef retrieve_examples(records,asset,embed_dir,max_labels=9):\n    cases=asset.get(\'cases\',[]); items=asset[\'items\']; comp=set(asset.get(\'competitive_codes\',[]))\n    query_specs=[]; priorities=[]\n    for ri,r in enumerate(records):\n        scores,_=candidate_scores(r,comp)\n        labs=select_priority_labels(scores,max_labels)\n        priorities.append(labs)\n        for v in labs: query_specs.append((ri,v,retrieval_query(r,v,items)))\n    examples=[{} for _ in records]\n    if not cases or not query_specs: return examples,priorities,{\'mode\':\'none\'}\n    report={\'mode\':RET_MODE,\'cases\':len(cases),\'queries\':len(query_specs),\'channels\':[]}\n    D=S=None\n    if RET_MODE in (\'hybrid\',\'dense\'):\n        try:\n            if not embed_dir or not Path(embed_dir).exists(): raise FileNotFoundError(embed_dir)\n            texts=[c[\'text\'] for c in cases]+[q for _,_,q in query_specs]\n            D,S=embed_texts(texts,embed_dir,max_length=RET_EMBED_MAX_LEN,want_sparse=(RET_MODE==\'hybrid\' and RET_W[\'sparse\']>0))\n            report[\'channels\'].append(\'dense\')\n            if S is not None: report[\'channels\'].append(\'sparse\')\n        except Exception as e:\n            log(f\'BGE retrieval failed -> lexical channels only: {type(e).__name__}: {e}\')\n            report[\'embed_error\']=str(e)[:200]; D=S=None\n    use_bm25=RET_MODE in (\'hybrid\',\'lexical\') or D is None\n    if use_bm25: report[\'channels\'].append(\'bm25\')\n    if RET_W[\'meta\']>0: report[\'channels\'].append(\'meta\')\n    nC=len(cases)\n    by_label={v:[] for v in ITEMS}\n    for i,c in enumerate(cases): by_label.setdefault(c[\'label\'],[]).append(i)\n    bm={v:(_BM25([cases[i][\'text\'] for i in idx]) if use_bm25 and idx else None) for v,idx in by_label.items()}\n    cmeta=[_case_meta(c[\'text\']) for c in cases]\n    rec_meta=[_meta_key(r.get(\'meta\',{})) for r in records]\n    def rrf(rank_lists):\n        sc={}\n        for ch,ranked in rank_lists:\n            w=RET_W[ch]\n            for rk,i in enumerate(ranked): sc[i]=sc.get(i,0.0)+w/(RET_RRF_K+rk+1)\n        return sc\n    for qi,(ri,v,qtext) in enumerate(query_specs):\n        idxs=[i for i in by_label.get(v,[]) if cases[i].get(\'source_id\')!=records[ri][\'id\']]\n        if not idxs: continue\n        pos_in_label={i:k for k,i in enumerate(by_label[v])}\n        qb=_bigrams(qtext) if use_bm25 else None\n        picked=[]\n        for y,need in ((1,RET_POS),(0,RET_NEG)):\n            ys=[i for i in idxs if int(cases[i][\'y\'])==y]\n            if not ys or need<=0: continue\n            chans=[]\n            if D is not None:\n                q=D[nC+qi]; chans.append((\'dense\',sorted(ys,key=lambda i:-float(np.dot(q,D[i])))))\n            if S is not None:\n                qs=S[nC+qi]; chans.append((\'sparse\',sorted(ys,key=lambda i:-_sparse_dot(qs,S[i]))))\n            if use_bm25:\n                chans.append((\'bm25\',sorted(ys,key=lambda i:-bm[v].score(qb,pos_in_label[i]))))\n            if RET_W[\'meta\']>0: chans.append((\'meta\',sorted(ys,key=lambda i:-_meta_sim(rec_meta[ri],cmeta[i]))))\n            fused=rrf(chans)\n            seen=set(); got=0\n            for i in sorted(ys,key=lambda i:-fused[i]):          # diversity: one example per source notice\n                sid=cases[i].get(\'source_id\')\n                if sid in seen: continue\n                seen.add(sid); picked.append(cases[i]); got+=1\n                if got>=need: break\n        if picked: examples[ri][v]=picked\n    return examples,priorities,report\n\ndef build_prompts(records,asset,examples,tokenizer,max_model_len,reserve_tokens):\n    # Strict context budget.\n    # Reserve enough room for the *larger* retry generation too.\n    sys_prompt=system_prompt(asset[\'items\'])\n    prompts=[]; stats=[]\n    limit=max_model_len-reserve_tokens-512\n    if limit <= 4096:\n        raise ValueError(f\'invalid prompt budget: model={max_model_len}, reserve={reserve_tokens}\')\n\n    # (doc_chars, law_labels, case_labels, case_chars): cover more items with shorter, evidence-centred cases\n    budget_plan=[\n        (7000,6,9,420),\n        (6000,5,8,400),\n        (5000,4,7,380),\n        (4200,4,6,350),\n        (3500,3,5,320),\n        (2900,2,4,300),\n        (2200,2,3,300),\n        (1700,1,2,300),\n        (1300,1,1,300),\n        (900,0,0,0),\n        (600,0,0,0),\n    ]\n\n    for i,r in enumerate(records):\n        prompt=None; ntok=None; meta=None; used=None\n\n        for doc_chars,law_n,case_n,case_chars in budget_plan:\n            user,meta=build_user_prompt(\n                r,asset,examples[i],\n                doc_chars=doc_chars,\n                max_law_labels=law_n,\n                max_case_labels=case_n,\n                case_chars=case_chars,\n            )\n            messages=[\n                {\'role\':\'system\',\'content\':sys_prompt},\n                {\'role\':\'user\',\'content\':user},\n            ]\n            try:\n                prompt=tokenizer.apply_chat_template(\n                    messages,tokenize=False,add_generation_prompt=True\n                )\n            except Exception:\n                prompt=sys_prompt+\'\\n\\n\'+user\n\n            ntok=len(tokenizer.encode(prompt,add_special_tokens=False))\n            used=(doc_chars,law_n,case_n)\n            if ntok <= limit:\n                break\n\n        if ntok is None or ntok > limit:\n            # Last-resort minimal prompt: no law/case examples, very small doc context.\n            user,meta=build_user_prompt(\n                r,asset,examples[i],\n                doc_chars=350,\n                max_law_labels=0,\n                max_case_labels=0,\n            )\n            messages=[\n                {\'role\':\'system\',\'content\':sys_prompt},\n                {\'role\':\'user\',\'content\':user},\n            ]\n            try:\n                prompt=tokenizer.apply_chat_template(\n                    messages,tokenize=False,add_generation_prompt=True\n                )\n            except Exception:\n                prompt=sys_prompt+\'\\n\\n\'+user\n            ntok=len(tokenizer.encode(prompt,add_special_tokens=False))\n            used=(350,0,0)\n\n        if ntok > limit:\n            raise RuntimeError(\n                f\'prompt budget overflow BEFORE vLLM id={r["id"]} \'\n                f\'tokens={ntok} limit={limit} reserve={reserve_tokens} \'\n                f\'last_budget={used}\'\n            )\n\n        prompts.append(prompt)\n        stats.append({\n            \'id\':r[\'id\'],\n            \'tokens\':ntok,\n            \'doc_chars\':used[0],\n            \'law_labels\':used[1],\n            \'case_labels\':used[2],\n            \'priority\':meta[\'priority\'],\n        })\n\n    return prompts,stats\n\ndef _zero_row(rec):\n    row={\'id\':rec[\'id\']}\n    for i in range(1,25): row[f\'v{i}\']=0; row[f\'e{i}\']=\'\'\n    return row\n\ndef _make_engine(args):\n    """Create the vLLM engine; on OOM / KV-cache shortfall retry with a shorter context (e.g. 24GB GPUs)."""\n    from vllm import LLM\n    import gc, torch\n    quant=None if str(args.quantization).lower() in (\'\',\'none\',\'auto\') else args.quantization\n    ladder=[args.max_model_len]+[x for x in (12288,8192) if x<args.max_model_len and args.auto_shrink]\n    last=None\n    for mml in ladder:\n        kw=dict(model=args.model_dir,tokenizer=args.model_dir,dtype=\'bfloat16\',quantization=quant,\n            max_model_len=mml,gpu_memory_utilization=args.gpu_mem,max_num_seqs=args.max_num_seqs,\n            max_num_batched_tokens=max(args.max_num_batched_tokens,1024),enable_chunked_prefill=True,\n            seed=args.seed,trust_remote_code=False,enforce_eager=args.enforce_eager)\n        if args.kv_cache_dtype and args.kv_cache_dtype!=\'auto\': kw[\'kv_cache_dtype\']=args.kv_cache_dtype\n        try:\n            log(f\'vLLM init max_model_len={mml} quant={quant} gpu_mem={args.gpu_mem} kv={args.kv_cache_dtype} eager={args.enforce_eager}\')\n            return LLM(**kw),mml\n        except Exception as e:\n            last=e; log(f\'vLLM init failed at max_model_len={mml}: {type(e).__name__}: {str(e)[:300]}\')\n            gc.collect()\n            try: torch.cuda.empty_cache()\n            except Exception: pass\n    raise last\n\ndef run(args):\n    started=time.monotonic()\n    records=load_records(args.input,args.limit)\n    if not records: raise ValueError(\'empty input\')\n    asset=json.loads(Path(args.asset).read_text(encoding=\'utf-8\'))\n    if args.mock:\n        rows=mock_rows(records); write_csv(rows,args.output); return {\'records\':len(rows),\'mode\':\'mock\'}\n    # --resume: rows finished in a previous (interrupted) run are restored from the checkpoint file\n    ckpt=Path(str(args.output)+\'.partial.jsonl\'); done={}\n    if args.resume and not ckpt.exists() and Path(args.output).exists():\n        # finished output from a previous session (e.g. Colab reconnect): reuse if it covers exactly these ids\n        with Path(args.output).open(encoding=\'utf-8\',newline=\'\') as f:\n            prev=[r[\'id\'] for r in csv.DictReader(f)]\n        if prev==[r[\'id\'] for r in records]:\n            log(f\'resume: {args.output} already complete ({len(prev)} rows) -> reused\')\n            rep={\'records\':len(prev),\'reused_existing_output\':True}\n            if args.report and not Path(args.report).exists():\n                Path(args.report).write_text(json.dumps(rep,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n            return rep\n    if args.resume and ckpt.exists():\n        for line in ckpt.read_text(encoding=\'utf-8\').splitlines():\n            if line.strip():\n                row=json.loads(line); done[row[\'id\']]=row\n        log(f\'resume: {len(done)} rows restored from {ckpt.name}\')\n    elif ckpt.exists():\n        ckpt.unlink()\n    todo=[r for r in records if r[\'id\'] not in done]\n    if todo:\n        examples,priorities,ret_report=retrieve_examples(todo,asset,args.embed_dir,args.max_case_labels)\n    else:\n        examples,priorities,ret_report=[],[],{\'mode\':\'skipped\'}\n    retrieved=time.monotonic()\n    if args.retrieval_only:\n        dump=[{\'id\':r[\'id\'],\'priority\':priorities[i],\n               \'examples\':{v:[{\'y\':c[\'y\'],\'source_id\':c.get(\'source_id\'),\'source\':c.get(\'source\'),\'text\':c[\'text\'][:300]} for c in arr]\n                           for v,arr in examples[i].items()}} for i,r in enumerate(todo)]\n        rep={\'records\':len(todo),\'retrieval\':ret_report,\'retrieval_seconds\':round(retrieved-started,1),\'examples\':dump}\n        if args.report: Path(args.report).write_text(json.dumps(rep,ensure_ascii=False,indent=1),encoding=\'utf-8\')\n        log(json.dumps(ret_report,ensure_ascii=False)); return rep\n    rows_new=[]; invalid_e=0; retries=0; failed=0; pstats=[{\'tokens\':0}]; initialized=retrieved; mml=args.max_model_len\n    if todo:\n        import torch, gc\n        from transformers import AutoTokenizer\n        from vllm import SamplingParams\n        from vllm.sampling_params import StructuredOutputsParams\n        gc.collect()\n        try:\n            torch.cuda.empty_cache(); torch.cuda.synchronize()\n        except Exception:\n            pass\n        engine,mml=_make_engine(args)                       # engine first: context length may shrink on small GPUs\n        initialized=time.monotonic()\n        tokenizer=AutoTokenizer.from_pretrained(args.model_dir,local_files_only=True)\n        reserve_tokens=max(args.max_tokens,args.retry_tokens)\n        prompts,pstats=build_prompts(todo,asset,examples,tokenizer,mml,reserve_tokens)\n        log(f\'prompt tokens median={int(np.median([x["tokens"] for x in pstats]))} max={max(x["tokens"] for x in pstats)} limit={mml-reserve_tokens-512} reserve={reserve_tokens}\')\n        def params(n):\n            return SamplingParams(temperature=0.0,max_tokens=n,seed=args.seed,\n                structured_outputs=StructuredOutputsParams(json=schema(),disable_any_whitespace=True))\n        with ckpt.open(\'a\',encoding=\'utf-8\') as ck:\n            for s in range(0,len(todo),args.batch_size):\n                batch_prompts=prompts[s:s+args.batch_size]\n                outs=engine.generate(batch_prompts,params(args.max_tokens),use_tqdm=False)\n                for r,p,o in zip(todo[s:s+args.batch_size],batch_prompts,outs):\n                    cand=o.outputs[0]\n                    try:\n                        if cand.finish_reason==\'length\': raise ValueError(\'length\')\n                        row,bad=parse_output(json.loads(cand.text),r)\n                    except Exception:\n                        retries+=1\n                        try:\n                            rr=engine.generate([p],params(args.retry_tokens),use_tqdm=False)[0].outputs[0]\n                            if rr.finish_reason==\'length\': raise ValueError(\'truncated twice\')\n                            row,bad=parse_output(json.loads(rr.text),r)\n                        except Exception as e:                  # never abort the whole run for one notice\n                            log(f\'output failed twice -> all-zero row: {r["id"]}: {type(e).__name__}\')\n                            row,bad=_zero_row(r),0; failed+=1\n                    rows_new.append(row); invalid_e+=bad\n                    ck.write(json.dumps(row,ensure_ascii=False)+\'\\n\')\n                ck.flush()\n                log(f\'{len(done)+len(rows_new)}/{len(records)} elapsed={time.monotonic()-started:.1f}s\')\n    by_id={**done,**{r[\'id\']:r for r in rows_new}}\n    rows=[by_id[r[\'id\']] for r in records]\n    write_csv(rows,args.output)\n    if ckpt.exists(): ckpt.unlink()\n    report={\'records\':len(rows),\'resumed_rows\':len(done),\'retrieval\':ret_report,\'retrieval_seconds\':round(retrieved-started,1),\n        \'initialization_seconds\':round(initialized-retrieved,1),\'processing_seconds\':round(time.monotonic()-initialized,1),\n        \'elapsed_seconds\':round(time.monotonic()-started,1),\'retries\':retries,\'failed_zero_rows\':failed,\'invalid_evidence_cleared\':invalid_e,\n        \'max_model_len_used\':mml,\n        \'prompt_token_median\':int(np.median([x[\'tokens\'] for x in pstats])),\'prompt_token_max\':max(x[\'tokens\'] for x in pstats)}\n    if args.report: Path(args.report).write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    log(json.dumps(report,ensure_ascii=False)); return report\n\ndef main():\n    ap=argparse.ArgumentParser()\n    ap.add_argument(\'--input\',default=os.path.join(os.environ.get(\'PPS_DATA_DIR\',\'./data\'),\'test.jsonl.gz\'))\n    ap.add_argument(\'--output\',default=os.path.join(os.environ.get(\'PPS_OUTPUT_DIR\',\'./output\'),\'submission.csv\'))\n    ap.add_argument(\'--asset\',default=str(DEFAULT_ASSET))\n    ap.add_argument(\'--model-dir\',default=os.environ.get(\'PPS_MODEL_DIR\',\'/opt/models/gemma-4-26B-A4B-it\'))\n    ap.add_argument(\'--embed-dir\',default=os.environ.get(\'PPS_EMBED_DIR\',\'/opt/models/bge-m3\'))\n    ap.add_argument(\'--retrieval-only\',action=\'store_true\',help=\'검색만 실행해 사례 선택 결과를 --report에 기록(모델 불필요)\')\n    ap.add_argument(\'--report\',default=None); ap.add_argument(\'--limit\',type=int,default=None); ap.add_argument(\'--mock\',action=\'store_true\')\n    ap.add_argument(\'--max-model-len\',type=int,default=16384); ap.add_argument(\'--max-tokens\',type=int,default=1536); ap.add_argument(\'--retry-tokens\',type=int,default=2048)\n    ap.add_argument(\'--gpu-mem\',type=float,default=0.88); ap.add_argument(\'--max-num-seqs\',type=int,default=4); ap.add_argument(\'--max-num-batched-tokens\',type=int,default=4096)\n    ap.add_argument(\'--batch-size\',type=int,default=8); ap.add_argument(\'--max-case-labels\',type=int,default=9); ap.add_argument(\'--seed\',type=int,default=42)\n    ap.add_argument(\'--quantization\',default=\'int8_per_channel_weight_only\')\n    ap.add_argument(\'--kv-cache-dtype\',default=os.environ.get(\'PPS_KV_CACHE_DTYPE\',\'auto\'))\n    ap.add_argument(\'--enforce-eager\',action=\'store_true\',default=os.environ.get(\'PPS_ENFORCE_EAGER\',\'0\')==\'1\')\n    ap.add_argument(\'--no-auto-shrink\',dest=\'auto_shrink\',action=\'store_false\',help=\'OOM 시 컨텍스트 자동 축소 끔\')\n    ap.add_argument(\'--resume\',action=\'store_true\',help=\'중단된 실행의 <output>.partial.jsonl 체크포인트에서 이어서 실행\')\n    args=ap.parse_args(); run(args)\nif __name__==\'__main__\': main()\n'
CORE_SOURCE = CORE_SOURCE.replace('os.environ.get("PPS_CONTEXTUAL","1")', f'os.environ.get("PPS_CONTEXTUAL","{int(CONTEXTUAL)}")')
CORE_SOURCE = CORE_SOURCE.replace('os.environ.get("PPS_CHUNKING","structure")', f'os.environ.get("PPS_CHUNKING","{CHUNKING}")')
(WORK / 'v2_core.py').write_text(CORE_SOURCE, encoding='utf-8')
(WORK / 'build_assets.py').write_text(BUILD_SOURCE, encoding='utf-8')
(WORK / 'script.py').write_text(SCRIPT_SOURCE, encoding='utf-8')
subprocess.run([str(PYTHON), '-m', 'py_compile', str(WORK / 'v2_core.py'), str(WORK / 'build_assets.py'), str(WORK / 'script.py')], check=True)
print('✅ Clean V2 코드 생성/컴파일 완료 — strict 16K token budget')

# ============================================================
# FINAL PIPELINE PATCH
# ============================================================
script_path = WORK/'script.py'
script_txt = script_path.read_text(encoding='utf-8')

_default_replacements = {
    "ap.add_argument('--max-model-len',type=int,default=16384)": f"ap.add_argument('--max-model-len',type=int,default={PROD_MAX_MODEL_LEN})",
    "ap.add_argument('--max-tokens',type=int,default=1536)": f"ap.add_argument('--max-tokens',type=int,default={PROD_MAX_TOKENS})",
    "ap.add_argument('--retry-tokens',type=int,default=2048)": f"ap.add_argument('--retry-tokens',type=int,default={PROD_RETRY_TOKENS})",
    "ap.add_argument('--gpu-mem',type=float,default=0.88)": f"ap.add_argument('--gpu-mem',type=float,default={PROD_GPU_MEM})",
    "ap.add_argument('--max-num-seqs',type=int,default=4)": f"ap.add_argument('--max-num-seqs',type=int,default={PROD_MAX_NUM_SEQS})",
    "ap.add_argument('--max-num-batched-tokens',type=int,default=4096)": f"ap.add_argument('--max-num-batched-tokens',type=int,default={PROD_MAX_BATCHED_TOKENS})",
    "ap.add_argument('--batch-size',type=int,default=8)": f"ap.add_argument('--batch-size',type=int,default={PROD_BATCH_SIZE})",
    "ap.add_argument('--max-case-labels',type=int,default=9)": f"ap.add_argument('--max-case-labels',type=int,default={PROD_MAX_CASE_LABELS})",
    "ap.add_argument('--seed',type=int,default=42)": f"ap.add_argument('--seed',type=int,default={PROD_SEED})",
    "ap.add_argument('--quantization',default='int8_per_channel_weight_only')": f"ap.add_argument('--quantization',default='{PROD_QUANTIZATION}')",
    # 하이브리드 검색 기본값(환경변수 PPS_RET_*로 덮어쓰기 가능)
    "os.environ.get('PPS_RET_MODE','hybrid')": f"os.environ.get('PPS_RET_MODE',{RET_MODE!r})",
    "os.environ.get('PPS_RET_MAX_LEN','1024')": f"os.environ.get('PPS_RET_MAX_LEN','{RET_MAX_LEN}')",
    "os.environ.get('PPS_RET_POS','1')": f"os.environ.get('PPS_RET_POS','{RET_POS}')",
    "os.environ.get('PPS_RET_NEG','1')": f"os.environ.get('PPS_RET_NEG','{RET_NEG}')",
    "os.environ.get('PPS_RET_W_DENSE','1.0')": f"os.environ.get('PPS_RET_W_DENSE','{RET_W_DENSE}')",
    "os.environ.get('PPS_RET_W_SPARSE','0.0')": f"os.environ.get('PPS_RET_W_SPARSE','{RET_W_SPARSE}')",
    "os.environ.get('PPS_RET_W_BM25','1.0')": f"os.environ.get('PPS_RET_W_BM25','{RET_W_BM25}')",
    "os.environ.get('PPS_RET_W_META','0.0')": f"os.environ.get('PPS_RET_W_META','{RET_W_META}')",
}

for old, new in _default_replacements.items():
    if old in script_txt:
        script_txt = script_txt.replace(old, new)

script_path.write_text(script_txt, encoding='utf-8')

ASSET_EXTRA_SOURCE = 'from __future__ import annotations\nimport argparse, gzip, hashlib, json\nfrom pathlib import Path\nfrom typing import Any, Dict\nfrom v2_core import ITEMS, ABSENCE, case_text\n\ndef _load(path: Path) -> Dict[str, Any]:\n    return json.loads(path.read_text(encoding="utf-8"))\n\ndef _save(obj: Dict[str, Any], path: Path) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(obj, ensure_ascii=False), encoding="utf-8")\n    print(json.dumps(obj.get("build_manifest", {}), ensure_ascii=False, indent=2))\n    print("asset MB", round(path.stat().st_size / 2**20, 2))\n\ndef law_only(base_asset: Path, out: Path) -> None:\n    """Dev 평가용: 같은 dev 정답 사례를 RAG에서 제거."""\n    asset = _load(base_asset)\n    n_gold = sum(1 for c in asset.get("cases", []) if c.get("source") == "gold")\n    asset["cases"] = []\n    bm = dict(asset.get("build_manifest", {}))\n    bm.update({\n        "gold_cases_available_but_excluded_for_dev_eval": n_gold,\n        "gold_cases": 0,\n        "audited_weak_cases": 0,\n        "pseudo_cases": 0,\n        "evaluation_asset": True,\n    })\n    asset["build_manifest"] = bm\n    asset["version"] = str(asset.get("version", "v2")) + "+law-only"\n    _save(asset, out)\n\ndef add_audited_weak(root: Path, audit_labels: Path, asset_path: Path,\n                     out: Path, cap_per_item: int = 80) -> None:\n    """감사 번들의 weak_candidate_mask=1인 양성만 사례은행으로 사용."""\n    asset = _load(asset_path)\n    chosen = {v: [] for v in ITEMS}\n\n    with gzip.open(audit_labels, "rt", encoding="utf-8") as f:\n        for line in f:\n            if not line.strip():\n                continue\n            ann = json.loads(line)\n            if ann.get("duplicate_of"):\n                continue\n            if "dev_text_overlap" in set(ann.get("flags") or []):\n                continue\n\n            labels = ann.get("labels") or []\n            mask = ann.get("weak_candidate_mask") or []\n            evidence = ann.get("evidence") or []\n            rid = ann["id"]\n\n            for i, m in enumerate(mask):\n                if not m or i >= len(labels) or int(labels[i]) != 1:\n                    continue\n                v = f"v{i+1}"\n                ev = evidence[i] if i < len(evidence) else ""\n                if v not in ABSENCE and not ev:\n                    continue\n\n                # 특정 ID 순서에 편향되지 않도록 deterministic hash 순서.\n                tie = int(hashlib.md5(f"{v}:{rid}".encode()).hexdigest()[:12], 16)\n                chosen[v].append((tie, rid, ev))\n\n    keep_ids = set()\n    for v in ITEMS:\n        chosen[v].sort(key=lambda x: x[0])\n        chosen[v] = chosen[v][:cap_per_item]\n        keep_ids.update(rid for _, rid, _ in chosen[v])\n\n    records = {}\n    with gzip.open(root/\'train_unlabeled.jsonl.gz\', \'rt\', encoding=\'utf-8\') as f:\n        for line in f:\n            if not line.strip():\n                continue\n            rec = json.loads(line)\n            if rec["id"] in keep_ids:\n                records[rec["id"]] = rec\n\n    add = []\n    items = asset["items"]\n    for v in ITEMS:\n        for _, rid, ev in chosen[v]:\n            rec = records.get(rid)\n            if rec is None:\n                continue\n            add.append({\n                "source_id": rid,\n                "label": v,\n                "y": 1,\n                "source": "audit_weak",\n                "text": case_text(rec, v, items, ev),\n            })\n\n    seen = {\n        (c.get("source_id"), c.get("label"), int(c.get("y", 0)))\n        for c in asset.get("cases", [])\n    }\n    add = [\n        c for c in add\n        if (c["source_id"], c["label"], c["y"]) not in seen\n    ]\n    asset.setdefault("cases", []).extend(add)\n\n    bm = dict(asset.get("build_manifest", {}))\n    bm["audited_weak_cases"] = len(add)\n    bm["audited_weak_source_records"] = len({c["source_id"] for c in add})\n    bm["audited_weak_by_item"] = {\n        v: sum(c["label"] == v for c in add)\n        for v in ITEMS\n        if any(c["label"] == v for c in add)\n    }\n    asset["build_manifest"] = bm\n    asset["version"] = str(asset.get("version", "v2")) + "+audit-weak"\n    _save(asset, out)\n\ndef add_gold(base_asset: Path, asset_path: Path, out: Path) -> None:\n    """최종 제출용: 제공된 dev gold few-shot 사례를 static asset에 추가."""\n    base = _load(base_asset)\n    asset = _load(asset_path)\n    gold = [c for c in base.get("cases", []) if c.get("source") == "gold"]\n\n    seen = {\n        (c.get("source_id"), c.get("label"), int(c.get("y", 0)))\n        for c in asset.get("cases", [])\n    }\n    add = [\n        c for c in gold\n        if (c.get("source_id"), c.get("label"), int(c.get("y", 0))) not in seen\n    ]\n    asset.setdefault("cases", []).extend(add)\n\n    bm = dict(asset.get("build_manifest", {}))\n    bm["final_gold_cases_added"] = len(add)\n    bm["final_total_cases"] = len(asset.get("cases", []))\n    bm["evaluation_asset"] = False\n    asset["build_manifest"] = bm\n    asset["version"] = str(asset.get("version", "v2")) + "+gold-final"\n    _save(asset, out)\n\ndef main():\n    ap = argparse.ArgumentParser()\n    sp = ap.add_subparsers(dest="cmd", required=True)\n\n    a = sp.add_parser("law-only")\n    a.add_argument("--base", required=True)\n    a.add_argument("--out", required=True)\n\n    a = sp.add_parser("audited")\n    a.add_argument("--root", required=True)\n    a.add_argument("--audit-labels", required=True)\n    a.add_argument("--asset", required=True)\n    a.add_argument("--out", required=True)\n    a.add_argument("--cap-per-item", type=int, default=80)\n\n    a = sp.add_parser("add-gold")\n    a.add_argument("--base", required=True)\n    a.add_argument("--asset", required=True)\n    a.add_argument("--out", required=True)\n\n    x = ap.parse_args()\n    if x.cmd == "law-only":\n        law_only(Path(x.base), Path(x.out))\n    elif x.cmd == "audited":\n        add_audited_weak(\n            Path(x.root), Path(x.audit_labels), Path(x.asset),\n            Path(x.out), x.cap_per_item\n        )\n    else:\n        add_gold(Path(x.base), Path(x.asset), Path(x.out))\n\nif __name__ == "__main__":\n    main()\n'
(WORK/'asset_extra.py').write_text(ASSET_EXTRA_SOURCE, encoding='utf-8')
CTX_LLM_SOURCE = '"""Contextual Retrieval (Anthropic, 2024) for the RAG case bank.\n\nFor every case chunk, Gemma writes a 1-2 sentence context that situates the chunk in its source notice;\nthe case\'s `CTX ...` line is replaced with it (the case is re-embedded / re-indexed at retrieval time, so both\nthe dense/sparse embeddings and BM25 become "contextual"). Runs offline on Colab; the enriched asset is a static\nfile, so the evaluation server needs nothing extra. Prompts put the whole document first so vLLM prefix caching\nreuses it across chunks of the same notice.\n"""\nfrom __future__ import annotations\nimport argparse, gzip, hashlib, json, sys, time\nfrom pathlib import Path\nfrom v2_core import nfc, meta_text\n\nPROMPT = """<document>\n{meta}\n\n{doc}\n</document>\n위 문서는 나라장터 입찰공고입니다. 아래는 이 문서에서 뽑은 발췌입니다.\n<chunk>\n{chunk}\n</chunk>\n이 발췌가 공고 전체에서 어떤 맥락에 있는지(어떤 계약·어느 조항·무엇을 규정하는지) 검색 정확도를 높이기 위한 짧고 간결한 설명을 한국어 1~2문장으로 쓰세요. 설명만 답하세요."""\n\ndef log(x): print(\'[ctx]\', x, file=sys.stderr, flush=True)\n\ndef load_needed(root: Path, ids):\n    out = {}\n    for name in (\'dev.jsonl.gz\', \'train_unlabeled.jsonl.gz\'):\n        p = root / name\n        if not p.exists(): continue\n        with gzip.open(p, \'rt\', encoding=\'utf-8\') as f:\n            for line in f:\n                if not line.strip(): continue\n                r = json.loads(line)\n                if r[\'id\'] in ids: out[r[\'id\']] = r\n        if len(out) >= len(ids): break\n    return out\n\ndef doc_view(rec, max_chars):\n    docs = sorted(rec.get(\'docs\', []), key=lambda d: 0 if d.get(\'type\') == \'공고문\' else 1)\n    t = \'\\n\\n\'.join(f"[{d.get(\'type\',\'기타\')}]\\n{nfc(d.get(\'text\',\'\'))}" for d in docs)\n    return t[:max_chars]\n\ndef split_case(text):\n    """case text = label line, META line, optional CTX line, body -> (label, meta, body)"""\n    lines = text.split(chr(10))\n    meta = next((l for l in lines[1:3] if l.startswith(\'META \')), \'\')\n    body = [l for l in lines[1:] if not l.startswith(\'META \') and not l.startswith(\'CTX \')]\n    return lines[0], meta, chr(10).join(body)\n\ndef rebuild(text, ctx):\n    label, meta, body = split_case(text)\n    ctx = \' \'.join(ctx.split())[:300]\n    return f"{label}\\n{meta}\\nCTX {ctx}\\n{body}"[:3400]\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--root\', required=True)\n    ap.add_argument(\'--assets\', nargs=\'+\', required=True, help=\'asset json files, updated in place\')\n    ap.add_argument(\'--model-dir\', default=\'\')\n    ap.add_argument(\'--quantization\', default=\'int8_per_channel_weight_only\')\n    ap.add_argument(\'--max-doc-chars\', type=int, default=5000)\n    ap.add_argument(\'--max-model-len\', type=int, default=8192)\n    ap.add_argument(\'--gpu-mem\', type=float, default=0.88)\n    ap.add_argument(\'--cache\', default=\'ctx_cache.json\')\n    ap.add_argument(\'--mock\', action=\'store_true\', help=\'no model: keep rule-based CTX (tests the plumbing)\')\n    x = ap.parse_args()\n    assets = {p: json.loads(Path(p).read_text(encoding=\'utf-8\')) for p in x.assets}\n    cache = json.loads(Path(x.cache).read_text(encoding=\'utf-8\')) if Path(x.cache).exists() else {}\n    todo = {}\n    for a in assets.values():\n        for c in a.get(\'cases\', []):\n            sid = c.get(\'source_id\')\n            if not sid: continue\n            k = hashlib.md5((sid + \'\\n\' + split_case(c[\'text\'])[2]).encode(\'utf-8\')).hexdigest()\n            c[\'_ctx_key\'] = k\n            if k not in cache: todo[k] = (sid, split_case(c[\'text\'])[2][:1500])\n    log(f\'cases needing context: {len(todo)} (cached {len(cache)})\')\n    if todo and not x.mock:\n        recs = load_needed(Path(x.root), {sid for sid, _ in todo.values()})\n        keys = [k for k, (sid, _) in todo.items() if sid in recs]\n        keys.sort(key=lambda k: todo[k][0])              # group by notice -> prefix-cache hits\n        from vllm import LLM, SamplingParams\n        from transformers import AutoTokenizer\n        tok = AutoTokenizer.from_pretrained(x.model_dir, local_files_only=True)\n        prompts = []\n        for k in keys:\n            sid, chunk = todo[k]; r = recs[sid]\n            msg = [{\'role\': \'user\', \'content\': PROMPT.format(meta=meta_text(r), doc=doc_view(r, x.max_doc_chars), chunk=chunk)}]\n            prompts.append(tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True))\n        quant = None if str(x.quantization).lower() in (\'\', \'none\', \'auto\') else x.quantization\n        llm = LLM(model=x.model_dir, tokenizer=x.model_dir, dtype=\'bfloat16\', quantization=quant,\n                  max_model_len=x.max_model_len, gpu_memory_utilization=x.gpu_mem, enable_prefix_caching=True, seed=42)\n        sp = SamplingParams(temperature=0.0, max_tokens=120, seed=42)\n        t = time.time()\n        for s in range(0, len(prompts), 64):\n            outs = llm.generate(prompts[s:s + 64], sp, use_tqdm=False)\n            for k, o in zip(keys[s:s + 64], outs):\n                txt = o.outputs[0].text.strip()\n                if txt: cache[k] = txt\n            log(f\'{min(s + 64, len(prompts))}/{len(prompts)} {time.time() - t:.0f}s\')\n            Path(x.cache).write_text(json.dumps(cache, ensure_ascii=False), encoding=\'utf-8\')\n    total = 0\n    for p, a in assets.items():\n        n = 0\n        for c in a.get(\'cases\', []):\n            k = c.pop(\'_ctx_key\', None)\n            if k and k in cache:\n                c[\'text\'] = rebuild(c[\'text\'], cache[k]); n += 1\n        bm = dict(a.get(\'build_manifest\', {})); bm[\'contextual_llm_cases\'] = n\n        a[\'build_manifest\'] = bm; total += n\n        Path(p).write_text(json.dumps(a, ensure_ascii=False), encoding=\'utf-8\')\n    log(f\'contextualised case texts written: {total}\')\n\nif __name__ == \'__main__\':\n    main()\n'
(WORK/'ctx_llm.py').write_text(CTX_LLM_SOURCE, encoding='utf-8')

subprocess.run([
    str(PYTHON), '-m', 'py_compile',
    str(WORK/'v2_core.py'),
    str(WORK/'build_assets.py'),
    str(WORK/'asset_extra.py'),
    str(WORK/'script.py'),
    str(WORK/'ctx_llm.py'),
], check=True)

print('✅ production 기본값 통일 + audit asset helper 컴파일 완료')


In [ ]:
script_txt = (WORK/'script.py').read_text(encoding='utf-8')
core_txt = (WORK/'v2_core.py').read_text(encoding='utf-8')

checks = {
    'reserve_tokens': 'reserve_tokens=max(args.max_tokens,args.retry_tokens)' in script_txt,
    'budget_plan': 'budget_plan=[' in script_txt,
    'core_max_law_labels': 'max_law_labels' in core_txt,
    'core_max_case_labels': 'max_case_labels' in core_txt,
    'server_default_batch': f"default={PROD_BATCH_SIZE}" in script_txt,
    'server_default_case_labels': f"default={PROD_MAX_CASE_LABELS}" in script_txt,
    'server_default_model_len': f"default={PROD_MAX_MODEL_LEN}" in script_txt,
    'asset_extra': (WORK/'asset_extra.py').exists(),
    'hybrid_retrieval': all(k in script_txt for k in ('class _BM25', 'def _sparse_dot', 'def _meta_sim', 'RET_RRF_K')),
    'hybrid_default_mode': f"os.environ.get('PPS_RET_MODE',{RET_MODE!r})" in script_txt,
    'contrastive_prompt': '비위반 근접 사례' in core_txt,
    'structure_chunking': 'def _split_structure' in core_txt and f'"PPS_CHUNKING","{CHUNKING}"' in core_txt,
    'resume_checkpoint': "'--resume'" in script_txt and 'partial.jsonl' in script_txt,
    'oom_auto_shrink': 'def _make_engine' in script_txt,
    'contextual_retrieval': 'def context_line' in core_txt and (WORK/'ctx_llm.py').exists(),
}
print(checks)
assert all(checks.values()), checks
print('✅ 제출용 코드 정적 검사 통과')


## 4-A. 라벨 도구 생성 (`human_labels.py`) — `unlabeled.csv` 사람/자동 행 → 사례

- `--select human`: 사람 확정 행 → 양성 전부 + 항목당 hard negative `HUMAN_NEG_PER_ITEM`
- `--select auto`: 자동 라벨 행 → 항목당 양성 `AUTO_POS_CAP_PER_ITEM`(규칙 점수 높은 순) + hard negative `AUTO_NEG_PER_ITEM`
- 근거 문구가 원문과 일치하면 그 주변(앞 150자·뒤 450자)을 사례 본문으로 사용, 빈칸은 사용 안 함
- dev와 본문이 같은 공고는 누수 방지를 위해 제외


In [ ]:
HUMAN_SOURCE = '"""Labels in unlabeled.csv format -> RAG case bank / validation set.\n\nunlabeled.csv holds all 20,000 train_unlabeled rows: the first rows are human-verified labels and the rest are\nrule-based auto labels. `--select human|auto|all` picks which rows to use:\n  human : ids listed (with any 0/1) in --human-ids (human_labels.csv), else the first --human-rows labelled rows\n  auto  : every other labelled row (weak labels -> capped positives per item, few hard negatives)\nBlank cells (undecided) are never used. Notices whose text equals a dev notice are skipped (leakage guard).\n"""\nfrom __future__ import annotations\nimport argparse, csv, gzip, hashlib, json\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Set\nfrom v2_core import ITEMS, ABSENCE, case_text, candidate_scores, clean_evidence, nfc\n\n\ndef _labelled(r):\n    return any((r.get(v) or \'\').strip() in (\'0\', \'1\') for v in ITEMS)\n\n\ndef read_labels(path: Path, select: str = \'all\', human_ids: Optional[Set[str]] = None, human_rows: int = 400) -> Dict[str, Dict[str, str]]:\n    with Path(path).open(\'r\', encoding=\'utf-8-sig\', newline=\'\') as f:\n        rd = csv.DictReader(f)\n        need = [\'id\'] + ITEMS + [f\'e{i}\' for i in range(1, 25)]\n        miss = [c for c in need if c not in (rd.fieldnames or [])]\n        if miss:\n            raise ValueError(f\'라벨 CSV 열 누락: {miss[:5]}\')\n        rows = [r for r in rd if _labelled(r)]\n    bad = [(r[\'id\'], v) for r in rows for v in ITEMS if (r[v] or \'\').strip() not in (\'0\', \'1\', \'\')]\n    if bad:\n        raise ValueError(f\'0/1/빈칸 외 값: {bad[:5]}\')\n    if human_ids is None:\n        human_ids = {r[\'id\'] for r in rows[:human_rows]}\n    if select == \'human\':\n        rows = [r for r in rows if r[\'id\'] in human_ids]\n    elif select == \'auto\':\n        rows = [r for r in rows if r[\'id\'] not in human_ids]\n    return {r[\'id\']: r for r in rows}\n\n\ndef read_human_ids(path: Optional[str]) -> Optional[Set[str]]:\n    if not path or not Path(path).exists():\n        return None\n    with Path(path).open(\'r\', encoding=\'utf-8-sig\', newline=\'\') as f:\n        return {r[\'id\'] for r in csv.DictReader(f) if _labelled(r)}\n\n\ndef _text_hash(rec: Dict[str, Any]) -> str:\n    return hashlib.md5(nfc("\\n".join(d.get("text", "") for d in rec.get("docs", []))).encode("utf-8")).hexdigest()\n\n\ndef _load_records(path: Path, ids=None) -> Dict[str, Dict[str, Any]]:\n    out = {}\n    with gzip.open(path, "rt", encoding="utf-8") as f:\n        for line in f:\n            if not line.strip():\n                continue\n            r = json.loads(line)\n            if ids is None or r["id"] in ids:\n                out[r["id"]] = r\n    return out\n\n\ndef add_cases(root: Path, labels_csv: Path, asset_path: Path, out: Path, neg_per_item: int = 8, select: str = \'human\',\n              human_ids: Optional[Set[str]] = None, human_rows: int = 400, pos_cap: int = 0, source: str = \'human\') -> None:\n    """Positives (all, or top `pos_cap` per item by rule score) + hard negatives (highest rule score among 0s)."""\n    asset = json.loads(Path(asset_path).read_text(encoding="utf-8"))\n    items, comp = asset["items"], set(asset.get("competitive_codes", []))\n    labels = read_labels(labels_csv, select, human_ids, human_rows)\n    recs = _load_records(root / "train_unlabeled.jsonl.gz", set(labels))\n    dev_hash = {_text_hash(r) for r in _load_records(root / "dev.jsonl.gz").values()}\n    skipped_missing = [i for i in labels if i not in recs]\n    skipped_dev = [i for i in labels if i in recs and _text_hash(recs[i]) in dev_hash]\n    use = {i: labels[i] for i in labels if i in recs and i not in skipped_dev}\n    score = {rid: candidate_scores(recs[rid], comp)[0] for rid in use}\n    tie = lambda v, rid: int(hashlib.md5(f"{v}:{rid}".encode()).hexdigest()[:8], 16)\n\n    add: List[Dict[str, Any]] = []\n    pos_by, neg_by, blank = {}, {}, 0\n    for v in ITEMS:\n        ek = "e" + v[1:]\n        pos = [rid for rid, row in use.items() if (row[v] or \'\').strip() == "1"]\n        pos.sort(key=lambda rid: (-score[rid][v], tie(v, rid)))\n        if pos_cap > 0:\n            pos = pos[:pos_cap]\n        for rid in pos:\n            ev = "" if v in ABSENCE else clean_evidence(use[rid].get(ek, ""), recs[rid])\n            add.append({"source_id": rid, "label": v, "y": 1, "source": source, "text": case_text(recs[rid], v, items, ev)})\n            pos_by[v] = pos_by.get(v, 0) + 1\n        negs = []\n        for rid, row in use.items():\n            x = (row[v] or \'\').strip()\n            if x == "":\n                blank += 1\n            elif x == "0":\n                negs.append((-score[rid][v], tie(v, rid), rid))\n        negs.sort()\n        for _, _, rid in negs[:neg_per_item]:\n            add.append({"source_id": rid, "label": v, "y": 0, "source": source, "text": case_text(recs[rid], v, items, "")})\n            neg_by[v] = neg_by.get(v, 0) + 1\n\n    seen = {(c.get("source_id"), c.get("label"), int(c.get("y", 0))) for c in asset.get("cases", [])}\n    add = [c for c in add if (c["source_id"], c["label"], c["y"]) not in seen]\n    asset.setdefault("cases", []).extend(add)\n    bm = dict(asset.get("build_manifest", {}))\n    bm.update({\n        f"{source}_label_records": len(use),\n        f"{source}_cases": len(add),\n        f"{source}_pos_by_item": pos_by,\n        f"{source}_neg_by_item": neg_by,\n        f"{source}_blank_cells_ignored": blank,\n        f"{source}_skipped_not_in_train": len(skipped_missing),\n        f"{source}_skipped_dev_overlap": len(skipped_dev),\n    })\n    if source == \'human\':          # backward-compatible keys used by the notebook\n        bm["human_cases"] = len(add)\n    asset["build_manifest"] = bm\n    asset["version"] = str(asset.get("version", "v2")) + f"+{source}"\n    out.parent.mkdir(parents=True, exist_ok=True)\n    out.write_text(json.dumps(asset, ensure_ascii=False), encoding="utf-8")\n    print(json.dumps({k: bm[k] for k in bm if k.startswith(source)}, ensure_ascii=False, indent=2))\n    print("asset MB", round(out.stat().st_size / 2**20, 2))\n\n\ndef make_holdout(root: Path, labels_csv: Path, out_jsonl: Path, select: str = \'human\',\n                 human_ids: Optional[Set[str]] = None, human_rows: int = 400) -> None:\n    """Input jsonl.gz of the selected labelled records (original order) for scoring."""\n    labels = read_labels(labels_csv, select, human_ids, human_rows)\n    n = 0\n    with gzip.open(root / "train_unlabeled.jsonl.gz", "rt", encoding="utf-8") as f, \\\n            gzip.open(out_jsonl, "wt", encoding="utf-8") as g:\n        for line in f:\n            if line.strip() and json.loads(line)["id"] in labels:\n                g.write(line if line.endswith("\\n") else line + "\\n"); n += 1\n    print("holdout records", n)\n\n\ndef export_selected(labels_csv: Path, out_csv: Path, select: str, human_ids, human_rows: int) -> None:\n    labels = read_labels(labels_csv, select, human_ids, human_rows)\n    cols = [\'id\'] + ITEMS + [f\'e{i}\' for i in range(1, 25)]\n    with Path(out_csv).open(\'w\', encoding=\'utf-8\', newline=\'\') as f:\n        w = csv.DictWriter(f, cols, extrasaction=\'ignore\'); w.writeheader(); w.writerows(labels.values())\n    print(\'exported\', len(labels), \'rows ->\', out_csv)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    sp = ap.add_subparsers(dest="cmd", required=True)\n    for name in ("add", "holdout", "export"):\n        p = sp.add_parser(name)\n        p.add_argument("--labels", required=True)\n        p.add_argument("--select", choices=["human", "auto", "all"], default="human")\n        p.add_argument("--human-ids", default="", help="human_labels.csv: 이 파일에 라벨이 있는 id를 사람 라벨로 간주")\n        p.add_argument("--human-rows", type=int, default=400, help="--human-ids가 없을 때 앞쪽 라벨 행 수")\n        if name != "export":\n            p.add_argument("--root", required=True)\n        if name == "add":\n            p.add_argument("--asset", required=True); p.add_argument("--out", required=True)\n            p.add_argument("--neg-per-item", type=int, default=8)\n            p.add_argument("--pos-cap", type=int, default=0, help="항목별 양성 사례 상한(0=전부)")\n            p.add_argument("--source", default="")\n        else:\n            p.add_argument("--out", required=True)\n    x = ap.parse_args()\n    hid = read_human_ids(x.human_ids)\n    if x.cmd == "add":\n        add_cases(Path(x.root), Path(x.labels), Path(x.asset), Path(x.out), x.neg_per_item, x.select, hid,\n                  x.human_rows, x.pos_cap, x.source or x.select)\n    elif x.cmd == "holdout":\n        make_holdout(Path(x.root), Path(x.labels), Path(x.out), x.select, hid, x.human_rows)\n    else:\n        export_selected(Path(x.labels), Path(x.out), x.select, hid, x.human_rows)\n\n\nif __name__ == "__main__":\n    main()\n'
(WORK/'human_labels.py').write_text(HUMAN_SOURCE, encoding='utf-8')
subprocess.run([str(PYTHON), '-m', 'py_compile', str(WORK/'human_labels.py')], check=True)
env = os.environ.copy(); env['PYTHONPATH'] = str(WORK)
# 채점용: 사람 확정 행만 뽑은 CSV
HUMAN_LABELS = WORK/'human_labels_selected.csv'
subprocess.run([str(PYTHON), str(WORK/'human_labels.py'), 'export', *LABEL_SELECT_ARGS, '--select', 'human',
                '--out', str(HUMAN_LABELS)], cwd=WORK, env=env, check=True)
print('✅ human_labels.py 생성/컴파일 완료')


## 5. Gold dev + 고정 법령 매핑 + 경쟁제품표 → Base 사례/RAG 자산 생성


In [ ]:
BASE_ASSET = WORK/'assets_base.json'
env = os.environ.copy(); env['PYTHONPATH'] = str(WORK)
subprocess.run([str(PYTHON),str(WORK/'build_assets.py'),'base',
    '--root',str(ROOT),'--out',str(BASE_ASSET)], cwd=WORK, env=env, check=True)
base_manifest = json.loads(BASE_ASSET.read_text(encoding='utf-8'))['build_manifest']
print(base_manifest)


## 5-A. dev 평가용 **법령-only 자산** 생성

기존 V2는 제공된 `dev_labels.csv`로 만든 gold 사례를 자산에 넣은 뒤 같은 dev를 평가할 수 있었습니다.  
그렇게 하면 dev 점수가 과대평가될 수 있으므로, 아래 평가용 자산에서는 **gold 사례를 전부 제거**합니다.


In [ ]:
EVAL_BASE_ASSET = WORK/'assets_eval_law_only.json'

subprocess.run([
    str(PYTHON), str(WORK/'asset_extra.py'), 'law-only',
    '--base', str(BASE_ASSET),
    '--out', str(EVAL_BASE_ASSET),
], cwd=WORK, env=env, check=True)

eval_manifest = json.loads(EVAL_BASE_ASSET.read_text(encoding='utf-8'))['build_manifest']
print(eval_manifest)
assert eval_manifest['gold_cases'] == 0


## 5-B. `unlabeled.csv` 자동 라벨(19,600행) → weak 사례은행

규칙 기반 자동 라벨은 사람 라벨보다 정확도가 낮아서(사람 라벨 대비 macro F1 약 0.39) **항목당 양성 상한 + 적은 hard negative**만 사례로 넣고, 채택 여부는 dev 점수로 결정합니다.
(선택) `USE_AUDIT_BUNDLE=True`면 이전 감사 번들 weak 후보도 추가합니다.


In [ ]:
WEAK_ASSET = WORK/'assets_eval_auto_weak.json'
env = os.environ.copy(); env['PYTHONPATH'] = str(WORK)
if USE_AUTO_LABELS:
    subprocess.run([str(PYTHON), str(WORK/'human_labels.py'), 'add', '--root', str(ROOT), *LABEL_SELECT_ARGS,
                    '--select', 'auto', '--source', 'auto', '--asset', str(EVAL_BASE_ASSET), '--out', str(WEAK_ASSET),
                    '--pos-cap', str(AUTO_POS_CAP_PER_ITEM), '--neg-per-item', str(AUTO_NEG_PER_ITEM)],
                   cwd=WORK, env=env, check=True)
else:
    shutil.copy2(EVAL_BASE_ASSET, WEAK_ASSET)
if AUDIT_LABELS is not None:
    subprocess.run([str(PYTHON), str(WORK/'asset_extra.py'), 'audited', '--root', str(ROOT), '--audit-labels', str(AUDIT_LABELS),
                    '--asset', str(WEAK_ASSET), '--out', str(WEAK_ASSET), '--cap-per-item', str(AUDIT_CASE_CAP_PER_ITEM)],
                   cwd=WORK, env=env, check=True)
weak_manifest = json.loads(WEAK_ASSET.read_text(encoding='utf-8'))['build_manifest']
print({k: v for k, v in weak_manifest.items() if not isinstance(v, dict)})


## 5-C. `unlabeled.csv` 사람 확정 행(400) → human 사례 추가

weak 자산(`WEAK_ASSET`)에 사람 확정 라벨 사례를 더한 `HUMAN_ASSET`을 만듭니다. 채택 여부는 dev 점수로 결정합니다.


In [ ]:
HUMAN_ASSET = WORK/'assets_eval_auto_human.json'
if USE_HUMAN_LABELS:
    subprocess.run([str(PYTHON), str(WORK/'human_labels.py'), 'add', '--root', str(ROOT), *LABEL_SELECT_ARGS,
                    '--select', 'human', '--source', 'human', '--asset', str(WEAK_ASSET), '--out', str(HUMAN_ASSET),
                    '--neg-per-item', str(HUMAN_NEG_PER_ITEM)], cwd=WORK, env=env, check=True)
    human_manifest = json.loads(HUMAN_ASSET.read_text(encoding='utf-8'))['build_manifest']
    assert human_manifest.get('human_cases', 0) > 0, '사람 라벨 사례가 0건입니다.'
else:
    HUMAN_ASSET = None
    print('USE_HUMAN_LABELS=False → human 자산 생략')


## 6. 모델 준비 — Gemma 4 26B(프로필별) + BGE-M3

- `a100`: 원본 `google/gemma-4-26B-A4B-it`(대회 고정 revision)을 받아 vLLM이 int8로 양자화하며 로드합니다(평가 서버와 동일).
- `rtx3090`: AWQ 4bit 커뮤니티 체크포인트.
- HF 토큰: Colab 보안 비밀 `HF_TOKEN` → 환경변수 `HF_TOKEN` → 입력창 순서로 찾습니다. Gemma는 HF에서 라이선스 동의가 필요합니다.
- 이미 받아둔 폴더가 있으면 `LOCAL_MODEL_DIR`, `LOCAL_EMBED_DIR`를 지정하면 다운로드를 건너뜁니다.


In [ ]:
from getpass import getpass
EMBED_ID = 'BAAI/bge-m3'
EMBED_REVISION = '5617a9f61b028005a4858fdac845db406aefb181'
EMBED_FILES = ['*.json', '*.model', 'pytorch_model.bin', 'sparse_linear.pt', 'colbert_linear.pt']

need_model = not (LOCAL_MODEL_DIR and Path(LOCAL_MODEL_DIR).exists())
need_embed = not (LOCAL_EMBED_DIR and Path(LOCAL_EMBED_DIR).exists())
if need_model or need_embed:
    hf_token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            hf_token = userdata.get('HF_TOKEN')
        except Exception:
            pass
    hf_token = hf_token or os.environ.get('HF_TOKEN') or (getpass('Hugging Face 읽기 토큰 (불필요하면 Enter): ').strip() or None)
    download_code = '''
import os, json
from pathlib import Path
from huggingface_hub import snapshot_download
tok = os.environ.get('HF_TOKEN') or None
out = {}
if os.environ.get('NEED_MODEL') == '1':
    out['model'] = snapshot_download(os.environ['MODEL_ID'], revision=os.environ.get('MODEL_REVISION') or None,
                                     cache_dir=os.environ['CACHE'], token=tok)
if os.environ.get('NEED_EMBED') == '1':
    out['embed'] = snapshot_download(os.environ['EMBED_ID'], revision=os.environ['EMBED_REVISION'], cache_dir=os.environ['CACHE'],
                                     token=tok, allow_patterns=json.loads(os.environ['EMBED_FILES']))
Path(os.environ['OUT']).write_text(json.dumps(out))
'''
    denv = os.environ.copy()
    if hf_token: denv['HF_TOKEN'] = hf_token
    denv.update(MODEL_ID=LOCAL_MODEL_ID, MODEL_REVISION=LOCAL_MODEL_REVISION or '', EMBED_ID=EMBED_ID, EMBED_REVISION=EMBED_REVISION,
                EMBED_FILES=json.dumps(EMBED_FILES), CACHE=str(HF_CACHE), OUT=str(WORK/'download_paths.json'),
                NEED_MODEL='1' if need_model else '0', NEED_EMBED='1' if need_embed else '0')
    subprocess.run([str(PYTHON), '-c', download_code], env=denv, check=True)
    paths = json.loads((WORK/'download_paths.json').read_text())
    del hf_token, denv
    if need_model: LOCAL_MODEL_DIR = paths['model']
    if need_embed: LOCAL_EMBED_DIR = paths['embed']
MODEL_DIR = Path(LOCAL_MODEL_DIR)
EMBED_DIR = Path(LOCAL_EMBED_DIR)
print('Gemma(로컬):', MODEL_DIR)
print('BGE-M3     :', EMBED_DIR, '| sparse head:', (EMBED_DIR/'sparse_linear.pt').exists())
qc = json.loads((MODEL_DIR/'config.json').read_text()).get('quantization_config', {})
print('checkpoint quantization_config:', qc.get('quant_method'), qc.get('bits'))


## 7. 서버형 환경변수 + mock 형식 테스트


In [ ]:
RUN_ENV = os.environ.copy()
RUN_ENV.update(
    PPS_DATA_DIR=str(ROOT/'data'),
    PPS_MODEL_DIR=str(MODEL_DIR),
    PPS_EMBED_DIR=str(EMBED_DIR),
    PPS_OUTPUT_DIR=str(WORK/'output'),
    HF_HUB_OFFLINE='1',
    TRANSFORMERS_OFFLINE='1',
    VLLM_WORKER_MULTIPROC_METHOD='spawn',
    PYTHONPATH=str(WORK),
)
RUN_ENV.pop('HF_TOKEN', None)
RUN_ENV['PATH'] = f"{Path(PYTHON).parent}:{RUN_ENV.get('PATH','')}"

# 로컬 GPU 서버(3090)에서 모든 추론에 쓰는 인자 — 제출물 기본값(PROD_*)과 별개
RUN_ARGS = [
    '--model-dir', str(MODEL_DIR), '--embed-dir', str(EMBED_DIR),
    '--quantization', LOCAL_QUANTIZATION, '--max-model-len', str(LOCAL_MAX_MODEL_LEN),
    '--gpu-mem', str(LOCAL_GPU_MEM), '--max-num-seqs', str(LOCAL_MAX_NUM_SEQS),
    '--batch-size', str(LOCAL_BATCH_SIZE), '--max-num-batched-tokens', str(LOCAL_MAX_BATCHED_TOKENS),
    '--kv-cache-dtype', LOCAL_KV_CACHE_DTYPE, '--max-case-labels', str(PROD_MAX_CASE_LABELS),
    '--max-tokens', str(PROD_MAX_TOKENS), '--retry-tokens', str(PROD_RETRY_TOKENS), '--seed', str(PROD_SEED),
    '--resume',
] + (['--enforce-eager'] if LOCAL_ENFORCE_EAGER else [])
PROD_ARGS = RUN_ARGS   # (이전 셀 호환용 이름)
print('RUN_ARGS:', ' '.join(RUN_ARGS))

def run_logged(cmd, log_path=None, env=RUN_ENV, cwd=WORK):
    """자식 프로세스 로그를 실시간 출력하고 실패 시 실제 traceback을 보여준다."""
    print("\n[RUN]", " ".join(map(str, cmd)))
    p = subprocess.Popen(
        cmd, cwd=cwd, env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    lines = []
    for line in p.stdout:
        print(line, end="")
        lines.append(line)
    rc = p.wait()
    if log_path:
        Path(log_path).write_text("".join(lines), encoding="utf-8")
    if rc != 0:
        tail = "".join(lines[-120:])
        raise RuntimeError(f"자식 프로세스 실패(exit={rc}). 마지막 로그:\n{tail}")
    return rc

run_logged(
    [
        str(PYTHON), str(WORK/'script.py'),
        '--mock',
        '--input', str(ROOT/'data'/'test.jsonl.gz'),
        '--output', str(WORK/'mock_submission.csv'),
        '--asset', str(BASE_ASSET),
    ],
    log_path=WORK/'mock_execution.log',
)
print("mock 완료:", WORK/'mock_submission.csv')


## 7-A. (선택) Contextual Retrieval — Gemma로 사례 문맥 생성

`CONTEXTUAL_LLM = True`일 때만 실행됩니다. Anthropic Contextual Retrieval 방식대로 사례 조각마다 **공고 전체를 보고** "이 발췌가 어떤 계약·어느 조항·무엇을 규정하는지" 1~2문장을 Gemma가 쓰고, 사례의 `CTX` 줄을 그 문장으로 바꿉니다.
- 같은 공고의 조각은 문서 부분이 같은 프롬프트 접두어라 vLLM prefix caching으로 재사용
- 결과는 자산(JSON)에 저장되는 정적 파일이라 평가 서버에서는 추가 연산 없음
- 생성 결과는 `ctx_cache.json`에 캐시되어 뒤 단계(최종 제출 자산)에서 재사용

In [ ]:
CTX_CACHE = WORK/'ctx_cache.json'
def contextualize(asset_paths):
    if not CONTEXTUAL_LLM:
        return
    paths = [str(p) for p in asset_paths if p is not None and Path(p).exists()]
    run_logged([str(PYTHON), str(WORK/'ctx_llm.py'), '--root', str(ROOT), '--assets', *paths,
                '--model-dir', str(MODEL_DIR), '--quantization', LOCAL_QUANTIZATION,
                '--max-doc-chars', str(CTX_MAX_DOC_CHARS), '--gpu-mem', str(LOCAL_GPU_MEM), '--cache', str(CTX_CACHE)],
               log_path=WORK/'ctx_llm_execution.log')

if CONTEXTUAL_LLM:
    contextualize([BASE_ASSET, EVAL_BASE_ASSET, WEAK_ASSET, globals().get('HUMAN_ASSET')])
    for p in [BASE_ASSET, EVAL_BASE_ASSET, WEAK_ASSET, globals().get('HUMAN_ASSET')]:
        if p is not None and Path(p).exists():
            print(Path(p).name, json.loads(Path(p).read_text(encoding='utf-8'))['build_manifest'].get('contextual_llm_cases'))
else:
    print('CONTEXTUAL_LLM=False → 규칙 기반 CTX만 사용 (CONTEXTUAL =', CONTEXTUAL, ')')


## 8. V2 Gold-only dev 200건 실제 추론


In [ ]:
# (이전 Colab 전용 패치 파일 %run 제거) — resume/체크포인트, 출력 실패 시 0행 대체, OOM 시 컨텍스트 자동 축소가 script.py에 내장됨
txt = (WORK/'script.py').read_text(encoding='utf-8')
print('resume/checkpoint:', "'--resume'" in txt and 'partial.jsonl' in txt)
print('출력 2회 실패 시 전체 중단 없음:', 'all-zero row' in txt)
print('OOM 시 컨텍스트 자동 축소:', 'def _make_engine' in txt)


In [ ]:
LAW_ONLY_DEV = WORK/'dev_law_only.csv'
run_logged([str(PYTHON), str(WORK/'script.py'), '--input', str(DEV_INPUT), '--output', str(LAW_ONLY_DEV),
            '--asset', str(EVAL_BASE_ASSET), '--report', str(WORK/'dev_law_only_run.json')] + RUN_ARGS,
           log_path=WORK/'dev_law_only_execution.log')


## 9. 공식 Macro F1 평가 함수 + Gold-only 점수


In [ ]:
import pandas as pd

def evaluate_submission(pred_path, label_path, title='DEV'):
    pred = pd.read_csv(pred_path)
    true = pd.read_csv(label_path)
    m = true[['id']+[f'v{i}' for i in range(1,25)]].merge(
        pred[['id']+[f'v{i}' for i in range(1,25)]], on='id', suffixes=('_true','_pred'))
    rows=[]
    for i in range(1,25):
        v=f'v{i}'; y=m[f'{v}_true'].astype(int); p=m[f'{v}_pred'].astype(int)
        tp=int(((y==1)&(p==1)).sum()); fp=int(((y==0)&(p==1)).sum()); fn=int(((y==1)&(p==0)).sum()); tn=int(((y==0)&(p==0)).sum())
        pr=tp/(tp+fp) if tp+fp else 0.0; rc=tp/(tp+fn) if tp+fn else 0.0; f1=2*pr*rc/(pr+rc) if pr+rc else 0.0
        rows.append(dict(item=v,positive_support=int((y==1).sum()),predicted_positive=int((p==1).sum()),tp=tp,fp=fp,fn=fn,tn=tn,precision=pr,recall=rc,f1=f1))
    df=pd.DataFrame(rows); score=float(df.f1.mean())
    print('='*70); print(f'{title} MACRO F1 = {score:.6f}'); print(f'V1 baseline = {V1_BASELINE_MACRO_F1:.6f} / delta = {score-V1_BASELINE_MACRO_F1:+.6f}'); print('='*70)
    display(df.sort_values('f1',ascending=False).reset_index(drop=True))
    return score,df

LAW_ONLY_SCORE, LAW_ONLY_METRICS = evaluate_submission(LAW_ONLY_DEV, ROOT/'dev_labels.csv', 'LAW-ONLY / NO DEV-RAG')


## 9-A. 자동 라벨 weak 사례 반영 후 dev 재평가

이 점수와 **LAW-ONLY** 점수를 비교하면 `unlabeled.csv` 자동 라벨이 RAG 사례로 도움이 되는지 확인할 수 있습니다.


In [ ]:
WEAK_DEV = WORK/'dev_auto_weak.csv'
run_logged([str(PYTHON), str(WORK/'script.py'), '--input', str(DEV_INPUT), '--output', str(WEAK_DEV),
            '--asset', str(WEAK_ASSET), '--report', str(WORK/'dev_auto_weak_run.json')] + RUN_ARGS,
           log_path=WORK/'dev_auto_weak_execution.log')
WEAK_SCORE, WEAK_METRICS = evaluate_submission(WEAK_DEV, ROOT/'dev_labels.csv', 'AUTO-WEAK RAG (unlabeled.csv 자동 라벨)')


## 9-B. 사람 확정 라벨 반영 후 dev 재평가 + 사람 라벨 검증셋

- **dev**: `AUTO-WEAK RAG` 점수와 비교해 사람 라벨이 실제로 도움이 되는지 확인합니다.
- **사람 라벨 검증셋**(선택): 사람 라벨 레코드 자체를 입력으로 채점합니다(`HOLDOUT_LIMIT`건). 같은 공고의 사례는 검색에서 제외되고, 빈칸 셀은 채점에서 뺍니다.
- 이후 pseudo 단계는 두 자산 중 dev 점수가 높은 쪽(`PRE_PSEUDO_ASSET`)을 기반으로 진행합니다.


In [ ]:
HUMAN_SCORE = None
HUMAN_DEV = WORK/'dev_auto_human.csv'
if USE_HUMAN_LABELS:
    run_logged([str(PYTHON), str(WORK/'script.py'), '--input', str(DEV_INPUT), '--output', str(HUMAN_DEV),
                '--asset', str(HUMAN_ASSET), '--report', str(WORK/'dev_auto_human_run.json')] + RUN_ARGS,
               log_path=WORK/'dev_auto_human_execution.log')
    HUMAN_SCORE, HUMAN_METRICS = evaluate_submission(HUMAN_DEV, ROOT/'dev_labels.csv', 'AUTO + HUMAN RAG')

# pseudo 단계의 기반 자산: dev 점수가 높은 쪽
if HUMAN_SCORE is not None and HUMAN_SCORE >= WEAK_SCORE:
    PRE_PSEUDO_ASSET, PRE_PSEUDO_DEV, PRE_PSEUDO_SCORE, PRE_PSEUDO_STAGE = HUMAN_ASSET, HUMAN_DEV, HUMAN_SCORE, 'auto_human'
else:
    PRE_PSEUDO_ASSET, PRE_PSEUDO_DEV, PRE_PSEUDO_SCORE, PRE_PSEUDO_STAGE = WEAK_ASSET, WEAK_DEV, WEAK_SCORE, 'auto_weak'
print('pseudo 기반 자산:', PRE_PSEUDO_STAGE, f'{PRE_PSEUDO_SCORE:.6f}')


In [ ]:
def evaluate_human(pred_path, label_path, title='HUMAN'):
    """사람 라벨 검증셋 채점. 빈칸(미확정) 셀은 제외하고, 양성이 있는 항목만 macro 평균."""
    pred = pd.read_csv(pred_path, dtype=str, keep_default_na=False).set_index('id')
    gold = pd.read_csv(label_path, dtype=str, keep_default_na=False).set_index('id')
    gold = gold[gold[[f'v{i}' for i in range(1,25)]].ne('').any(axis=1)]
    rows = []
    for i in range(1,25):
        v = f'v{i}'; tp = fp = fn = 0
        for rid, g in gold[v].items():
            if g == '' or rid not in pred.index:
                continue
            y, p = int(g), int(pred.at[rid, v])
            tp += y and p; fp += (not y) and p; fn += y and (not p)
        pr = tp/(tp+fp) if tp+fp else 0.0; rc = tp/(tp+fn) if tp+fn else 0.0
        rows.append(dict(item=v, support=tp+fn, tp=tp, fp=fp, fn=fn, precision=pr, recall=rc,
                         f1=2*pr*rc/(pr+rc) if pr+rc else 0.0))
    df = pd.DataFrame(rows); sup = df[df.support > 0]
    score = float(sup.f1.mean()) if len(sup) else 0.0
    print('='*70); print(f'{title} macro F1 (양성 있는 {len(sup)}개 항목) = {score:.6f}'); print('='*70)
    display(df[df.support.gt(0) | df.fp.gt(0)].reset_index(drop=True))
    return score, df

HOLDOUT_SCORES = {}
if USE_HUMAN_LABELS and EVAL_HUMAN_HOLDOUT:
    HOLDOUT_INPUT = WORK/'human_holdout.jsonl.gz'
    subprocess.run([str(PYTHON), str(WORK/'human_labels.py'), 'holdout', '--root', str(ROOT), *LABEL_SELECT_ARGS,
                    '--select', 'human', '--out', str(HOLDOUT_INPUT)], cwd=WORK, env=RUN_ENV, check=True)
    lim = ['--limit', str(HOLDOUT_LIMIT)] if HOLDOUT_LIMIT else []
    for tag, asset in [('auto_weak', WEAK_ASSET), ('auto_human', HUMAN_ASSET)]:
        out = WORK/f'holdout_{tag}.csv'
        run_logged([str(PYTHON), str(WORK/'script.py'), '--input', str(HOLDOUT_INPUT), '--output', str(out),
                    '--asset', str(asset), '--report', str(WORK/f'holdout_{tag}_run.json')] + lim + RUN_ARGS,
                   log_path=WORK/f'holdout_{tag}_execution.log')
        HOLDOUT_SCORES[tag], _ = evaluate_human(out, HUMAN_LABELS, f'HUMAN HOLDOUT · {tag}')
    print('HUMAN HOLDOUT:', {k: round(v, 6) for k, v in HOLDOUT_SCORES.items()})


## 9-C. (선택) 검색 방식 A/B — 하이브리드 vs 기존 dense

`COMPARE_RETRIEVAL = True`일 때만 실행됩니다. 같은 자산(`PRE_PSEUDO_ASSET`)으로 `PPS_RET_MODE=dense`(기존 방식)를 한 번 더 돌려 dev Macro F1을 비교합니다.
하이브리드가 더 낮게 나오면 1번 셀에서 `RET_MODE='dense'`로 바꾸고 4번부터 다시 실행하세요.

In [ ]:
RETRIEVAL_AB = {}
if COMPARE_RETRIEVAL:
    RETRIEVAL_AB[RET_MODE] = float(PRE_PSEUDO_SCORE)
    alt_mode = 'dense' if RET_MODE != 'dense' else 'hybrid'
    alt_env = RUN_ENV.copy(); alt_env['PPS_RET_MODE'] = alt_mode
    ALT_DEV = WORK/f'dev_retrieval_{alt_mode}.csv'
    run_logged([str(PYTHON), str(WORK/'script.py'), '--input', str(DEV_INPUT), '--output', str(ALT_DEV),
                '--asset', str(PRE_PSEUDO_ASSET), '--report', str(WORK/f'dev_retrieval_{alt_mode}_run.json')] + PROD_ARGS,
               log_path=WORK/f'dev_retrieval_{alt_mode}_execution.log', env=alt_env)
    RETRIEVAL_AB[alt_mode], _ = evaluate_submission(ALT_DEV, ROOT/'dev_labels.csv', f'RETRIEVAL={alt_mode}')
    print('검색 방식 A/B (dev Macro F1):', {k: round(v, 6) for k, v in RETRIEVAL_AB.items()})
    best = max(RETRIEVAL_AB, key=RETRIEVAL_AB.get)
    if best != RET_MODE:
        print(f'⚠️ {best}가 더 높습니다. 1번 셀 RET_MODE={best!r}로 바꾸고 4번 셀부터 다시 실행하는 것을 권장합니다.')
else:
    print('COMPARE_RETRIEVAL=False → A/B 생략 (현재 검색 방식:', RET_MODE, ')')


## 10. 20,000 무라벨 전체 스캔 → 항목별 pseudo-label 후보 mining

20,000건을 모두 Gemma로 24항목 판정하면 너무 오래 걸리므로, **전체 20k는 rule/meta로 전수 스캔**하고 label별 상위 후보만 실제 Gemma로 검증합니다. 이 단계는 자가 라벨링용 후보를 고르는 것이며 rule score 자체를 정답으로 사용하지 않습니다.


In [ ]:
PSEUDO_INPUT = WORK/'pseudo_candidates.jsonl.gz'
PSEUDO_MAP = WORK/'pseudo_map.json'

if RUN_PSEUDO:
    subprocess.run([str(PYTHON),str(WORK/'build_assets.py'),'mine',
        '--root',str(ROOT),'--asset',str(PRE_PSEUDO_ASSET),'--out',str(PSEUDO_INPUT),'--map',str(PSEUDO_MAP),
        '--per-item',str(PSEUDO_PER_ITEM)], cwd=WORK, env=RUN_ENV, check=True)
else:
    print('RUN_PSEUDO=False: pseudo mining 생략')


## 11. 후보만 Gemma+BGE로 자가 라벨링


In [ ]:
PSEUDO_PRED = WORK/'pseudo_submission.csv'
if RUN_PSEUDO:
    run_logged([str(PYTHON), str(WORK/'script.py'), '--input', str(PSEUDO_INPUT), '--output', str(PSEUDO_PRED),
                '--asset', str(PRE_PSEUDO_ASSET), '--report', str(WORK/'pseudo_run.json')] + RUN_ARGS,
               log_path=WORK/'pseudo_execution.log')
    print('pseudo prediction:', PSEUDO_PRED)
else:
    print('pseudo 추론 생략')


## 12. High-confidence pseudo positive만 사례은행에 추가


In [ ]:
FINAL_ASSET_DIR = WORK/'submit'/'model'
FINAL_ASSET_DIR.mkdir(parents=True,exist_ok=True)
FINAL_ASSET = FINAL_ASSET_DIR/'assets.json'

if RUN_PSEUDO:
    subprocess.run([str(PYTHON),str(WORK/'build_assets.py'),'augment',
        '--root',str(ROOT),'--asset',str(PRE_PSEUDO_ASSET),'--pred',str(PSEUDO_PRED),
        '--map',str(PSEUDO_MAP),'--candidates',str(PSEUDO_INPUT),'--out',str(FINAL_ASSET),
        '--max-per-item',str(PSEUDO_KEEP_PER_ITEM)], cwd=WORK, env=RUN_ENV, check=True)
else:
    shutil.copy2(PRE_PSEUDO_ASSET, FINAL_ASSET)

manifest=json.loads(FINAL_ASSET.read_text(encoding='utf-8'))['build_manifest']
print('FINAL ASSET:', manifest)


## 13. Final V2 dev 재평가 (pseudo 사례 포함)


In [ ]:
FINAL_DEV = WORK/'dev_v2_final.csv'
if RUN_PSEUDO:
    run_logged([str(PYTHON), str(WORK/'script.py'), '--input', str(DEV_INPUT), '--output', str(FINAL_DEV),
                '--asset', str(FINAL_ASSET), '--report', str(WORK/'dev_v2_final_run.json')] + RUN_ARGS,
               log_path=WORK/'dev_v2_final_execution.log')
else:
    shutil.copy2(PRE_PSEUDO_DEV, FINAL_DEV)

FINAL_SCORE, FINAL_METRICS = evaluate_submission(FINAL_DEV, ROOT/'dev_labels.csv', 'V2 FINAL')
FINAL_METRICS.to_csv(WORK/'dev_v2_final_metrics.csv', index=False, encoding='utf-8')


## 13-A. dev 결과 비교 → 제출 자산 선택 → dev gold few-shot 추가

선택 기준은 **dev Macro-F1**입니다.

- pseudo를 넣고 점수가 내려가면 감사 weak 자산으로 되돌립니다.
- 최종 제출 자산에는 제공된 dev gold 사례를 static few-shot으로 추가할 수 있습니다.
- 단, 그 gold 사례가 들어간 자산으로 다시 같은 dev 점수를 측정하지 않습니다.


In [ ]:
scores = {
    'law_only': LAW_ONLY_SCORE,
    'auto_weak': WEAK_SCORE,
}
if HUMAN_SCORE is not None:
    scores['auto_human'] = HUMAN_SCORE
if RUN_PSEUDO:
    scores[PRE_PSEUDO_STAGE + '_plus_pseudo'] = FINAL_SCORE
print('DEV SCOREBOARD')
for k,v in scores.items():
    print(f'  {k:18s}: {v:.6f}')

if RUN_PSEUDO and FINAL_SCORE >= PRE_PSEUDO_SCORE:
    CHOSEN_EVAL_ASSET = FINAL_ASSET
    CHOSEN_STAGE = PRE_PSEUDO_STAGE + '_plus_pseudo'
else:
    CHOSEN_EVAL_ASSET = PRE_PSEUDO_ASSET
    CHOSEN_STAGE = PRE_PSEUDO_STAGE

print('chosen stage:', CHOSEN_STAGE)

SUBMIT_ASSET = WORK/'submit'/'model'/'assets.json'
SUBMIT_ASSET.parent.mkdir(parents=True, exist_ok=True)

if ADD_DEV_GOLD_TO_SUBMIT:
    subprocess.run([
        str(PYTHON), str(WORK/'asset_extra.py'), 'add-gold',
        '--base', str(BASE_ASSET),
        '--asset', str(CHOSEN_EVAL_ASSET),
        '--out', str(SUBMIT_ASSET),
    ], cwd=WORK, env=RUN_ENV, check=True)
else:
    shutil.copy2(CHOSEN_EVAL_ASSET, SUBMIT_ASSET)

contextualize([SUBMIT_ASSET])   # 뒤에 추가된 gold/pseudo 사례도 Gemma 문맥 적용(캐시 재사용)
submit_manifest = json.loads(SUBMIT_ASSET.read_text(encoding='utf-8'))['build_manifest']
print('SUBMIT ASSET:', submit_manifest)

RESULT_SUMMARY = {
    'law_only_macro_f1': float(LAW_ONLY_SCORE),
    'auto_weak_macro_f1': float(WEAK_SCORE),
    'auto_human_macro_f1': (float(HUMAN_SCORE) if HUMAN_SCORE is not None else None),
    'plus_pseudo_macro_f1': float(FINAL_SCORE),
    'pseudo_base_stage': PRE_PSEUDO_STAGE,
    'labels': {'unlabeled_csv': str(UNLABELED_CSV.name), 'human_rows': len(human_ids), 'auto_pos_cap': AUTO_POS_CAP_PER_ITEM,
               'auto_neg': AUTO_NEG_PER_ITEM, 'human_neg': HUMAN_NEG_PER_ITEM, 'use_auto': USE_AUTO_LABELS, 'use_human': USE_HUMAN_LABELS},
    'local_runtime': {'model': LOCAL_MODEL_ID, 'quantization': LOCAL_QUANTIZATION, 'max_model_len': LOCAL_MAX_MODEL_LEN, 'dev_limit': DEV_LIMIT},
    'retrieval': {'mode': RET_MODE, 'max_len': RET_MAX_LEN, 'pos': RET_POS, 'neg': RET_NEG,
                  'weights': {'dense': RET_W_DENSE, 'sparse': RET_W_SPARSE, 'bm25': RET_W_BM25, 'meta': RET_W_META},
                  'ab_dev_macro_f1': RETRIEVAL_AB, 'contextual': CONTEXTUAL, 'contextual_llm': CONTEXTUAL_LLM, 'chunking': CHUNKING},
    'human_holdout_macro_f1': {k: float(v) for k, v in HOLDOUT_SCORES.items()},
    'chosen_stage': CHOSEN_STAGE,
    'dev_gold_added_only_after_selection': bool(ADD_DEV_GOLD_TO_SUBMIT),
}
(WORK/'pipeline_result_summary.json').write_text(
    json.dumps(RESULT_SUMMARY, ensure_ascii=False, indent=2),
    encoding='utf-8'
)
print(json.dumps(RESULT_SUMMARY, ensure_ascii=False, indent=2))


## 14. 공개 sample test를 서버 기본 경로와 동일하게 실행


In [ ]:
SAMPLE_OUT = WORK/'sample_output'
SAMPLE_OUT.mkdir(exist_ok=True)
sample_env = RUN_ENV.copy()
sample_env['PPS_OUTPUT_DIR'] = str(SAMPLE_OUT)
# 입력/출력 경로는 서버와 같이 PPS_DATA_DIR / PPS_OUTPUT_DIR 기본값을 쓰고, 런타임만 로컬(3090) 설정
run_logged([str(PYTHON), str(WORK/'script.py'), '--asset', str(SUBMIT_ASSET), '--report', str(WORK/'sample_v2_run.json')]
           + [x for x in RUN_ARGS if x != '--resume'],
           log_path=WORK/'sample_v2_execution.log', env=sample_env)

sample_csv = SAMPLE_OUT/'submission.csv'
assert sample_csv.exists()
sm = pd.read_csv(sample_csv)
assert list(sm.columns) == (['id'] + [f'v{i}' for i in range(1,25)] + [f'e{i}' for i in range(1,25)])
print('sample rows:', len(sm), 'columns:', len(sm.columns))
display(sm.head())


## 14-A. 대회 제출 형식 strict validator

공개 sample 추론 결과에 대해 대회 형식 조건을 먼저 검사합니다.


In [ ]:
import unicodedata

ABSENCE_E = {'e10','e11','e16','e18','e20'}
EXPECTED_COLS = ['id'] + [f'v{i}' for i in range(1,25)] + [f'e{i}' for i in range(1,25)]

def validate_submission_csv(csv_path, input_path):
    df = pd.read_csv(csv_path, dtype={f'e{i}':'string' for i in range(1,25)}, keep_default_na=False)

    assert list(df.columns) == EXPECTED_COLS, '49열/열 순서 불일치'
    assert len(df) == count_jsonl_gz(input_path), '입력/출력 행 수 불일치'
    assert df['id'].is_unique, 'id 중복'
    assert not df['id'].eq('').any(), '빈 id'

    with gzip.open(input_path,'rt',encoding='utf-8') as f:
        input_ids = [json.loads(line)['id'] for line in f if line.strip()]
    assert set(df['id']) == set(input_ids), '입력 id 누락/추가'

    for i in range(1,25):
        v = f'v{i}'
        vals = set(pd.to_numeric(df[v], errors='raise').astype(int).tolist())
        assert vals <= {0,1}, f'{v}에 0/1 외 값: {vals}'

    # 근거 원문 substring + 길이 + 부재탐지 빈칸
    records = {}
    with gzip.open(input_path,'rt',encoding='utf-8') as f:
        for line in f:
            if line.strip():
                r = json.loads(line)
                records[r['id']] = r

    for _, row in df.iterrows():
        rec = records[row['id']]
        full = '\\n'.join(
            unicodedata.normalize('NFC', str(d.get('text','')).replace('\\r',''))
            for d in rec.get('docs',[])
        )
        for i in range(1,25):
            e = f'e{i}'
            ev = str(row[e] or '')
            assert len(ev) <= 500, f'{row["id"]} {e} > 500자'
            if e in ABSENCE_E:
                assert ev == '', f'{row["id"]} {e} 부재탐지는 빈칸이어야 함'
            if int(row[f'v{i}']) == 0:
                assert ev == '', f'{row["id"]} v{i}=0인데 {e}가 비어있지 않음'
            if ev:
                assert not ev.startswith(('=','+','@')), f'{row["id"]} {e} 위험 접두문자'
                assert unicodedata.normalize('NFC',ev) in full, f'{row["id"]} {e}가 원문 substring 아님'

    raw = Path(csv_path).read_bytes()
    assert not raw.startswith(b'\\xef\\xbb\\xbf'), 'UTF-8 BOM 금지'
    print('✅ submission.csv strict validation 통과:', len(df), 'rows x', len(df.columns), 'cols')
    return df

validated_sample = validate_submission_csv(
    sample_csv,
    ROOT/'data'/'test.jsonl.gz'
)


## 15. `submit.zip` 생성

제출 ZIP에는 **실행 코드 + 제공 데이터로부터 미리 만든 정적 자산만** 넣습니다. Gemma/BGE 가중치, HF 토큰, dev_labels.csv, dev 예측 CSV는 포함하지 않습니다. 평가 서버에서는 `PPS_MODEL_DIR`, `PPS_EMBED_DIR`, `PPS_DATA_DIR`, `PPS_OUTPUT_DIR`을 사용합니다.


In [ ]:
STAGE = WORK/'submit'
STAGE.mkdir(parents=True, exist_ok=True)
(STAGE/'model').mkdir(parents=True, exist_ok=True)

# 최종 자산은 앞 단계에서 이미 STAGE/model/assets.json에 생성됨
assert SUBMIT_ASSET == STAGE/'model'/'assets.json'
assert SUBMIT_ASSET.exists()

shutil.copy2(WORK/'script.py', STAGE/'script.py')
shutil.copy2(WORK/'v2_core.py', STAGE/'v2_core.py')

# 평가 서버에 vllm/torch/transformers/xgrammar가 고정 제공되므로
# 별도 pip 설치가 필요하지 않은 제출을 기본으로 둠.
(STAGE/'requirements.txt').write_text(
    '# 평가 서버 기본 패키지 사용\n',
    encoding='utf-8'
)

# 재현 메타데이터: 실행에는 필요 없지만 정적 JSON이라 함께 보관 가능
submission_manifest = {
    'competition': '236754',
    'fixed_llm': 'google/gemma-4-26B-A4B-it',
    'fixed_llm_revision': '4d7ae4984b7db7de8f8457170b3f1a419ee76d52',
    'embed_model': 'BAAI/bge-m3',
    'embed_revision': '5617a9f61b028005a4858fdac845db406aefb181',
    'chosen_stage': CHOSEN_STAGE,
    'human_labels_used': bool(USE_HUMAN_LABELS and 'human' in CHOSEN_STAGE),
    'auto_labels_used': bool(USE_AUTO_LABELS),
    'runtime_defaults': {
        'max_model_len': PROD_MAX_MODEL_LEN,
        'max_tokens': PROD_MAX_TOKENS,
        'retry_tokens': PROD_RETRY_TOKENS,
        'batch_size': PROD_BATCH_SIZE,
        'max_num_seqs': PROD_MAX_NUM_SEQS,
        'max_num_batched_tokens': PROD_MAX_BATCHED_TOKENS,
        'max_case_labels': PROD_MAX_CASE_LABELS,
        'gpu_mem': PROD_GPU_MEM,
        'seed': PROD_SEED,
        'quantization': PROD_QUANTIZATION,
    },
    'dev_scores_before_gold_fewshot': RESULT_SUMMARY,
}
(STAGE/'model'/'pipeline_manifest.json').write_text(
    json.dumps(submission_manifest, ensure_ascii=False, indent=2),
    encoding='utf-8'
)

# 금지된 가중치 파일이 섞이지 않았는지 확장자로 사전검사
FORBIDDEN_EXT = {
    '.pt','.pth','.bin','.ckpt','.safetensors','.onnx',
    '.gguf','.h5','.joblib','.pkl','.pickle'
}
bad = [p for p in STAGE.rglob('*') if p.is_file() and p.suffix.lower() in FORBIDDEN_EXT]
assert not bad, f'학습 가중치/직렬화 모델 의심 파일 발견: {bad}'

SUBMIT_ZIP = WORK/'submit.zip'
zip_files = [
    'script.py',
    'v2_core.py',
    'requirements.txt',
    'model/assets.json',
    'model/pipeline_manifest.json',
]

with zipfile.ZipFile(SUBMIT_ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    for rel in zip_files:
        z.write(STAGE/rel, rel)

with zipfile.ZipFile(SUBMIT_ZIP) as z:
    names = z.namelist()
    print('ZIP contents:', names)
    assert 'script.py' in names
    assert 'requirements.txt' in names
    assert 'model/assets.json' in names
    assert all(not n.startswith('/') for n in names)
    assert not any(n.split('/')[0] == 'submit' for n in names)

print('submit.zip MB:', round(SUBMIT_ZIP.stat().st_size/2**20, 2))
assert SUBMIT_ZIP.stat().st_size < 2 * 1024**3
print('✅ submit.zip 구조 검사 통과')


## 16. 제출 ZIP · 실험 결과 보관 (Drive/서버 경로 저장 + Colab 다운로드)


In [ ]:
EXP = WORK/'pipeline_experiment_results.zip'
keep = ['dev_law_only.csv', 'dev_law_only_run.json', 'dev_auto_weak.csv', 'dev_auto_weak_run.json',
        'dev_auto_human.csv', 'dev_auto_human_run.json', 'holdout_auto_weak.csv', 'holdout_auto_human.csv',
        'dev_v2_final.csv', 'dev_v2_final_run.json', 'dev_v2_final_metrics.csv', 'sample_v2_run.json',
        'pseudo_run.json', 'pseudo_map.json', 'pipeline_result_summary.json']
with zipfile.ZipFile(EXP, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in keep:
        p = WORK/name
        if p.exists():
            z.write(p, name)
# 찾기 쉽게 WORK_BASE에도 최신본 복사
shutil.copy2(SUBMIT_ZIP, WORK_BASE/'submit_latest.zip')
shutil.copy2(EXP, WORK_BASE/'pipeline_experiment_results_latest.zip')
print('제출 파일      :', SUBMIT_ZIP, '→', WORK_BASE/'submit_latest.zip')
print('실험 결과 묶음 :', EXP, '→', WORK_BASE/'pipeline_experiment_results_latest.zip')

if IN_COLAB:
    from google.colab import files
    files.download(str(SUBMIT_ZIP))
    files.download(str(EXP))


## 실행 결과 해석

| 단계 | 사례은행 | 비교 의미 |
|---|---|---|
| LAW-ONLY | 법령·규칙만(사례 없음) | 기준점 |
| AUTO-WEAK | + `unlabeled.csv` 자동 라벨 사례(항목당 양성 ≤40, 음성 4) | LAW-ONLY보다 높으면 자동 라벨이 도움 |
| AUTO + HUMAN | + 사람 확정 400행 사례 | AUTO-WEAK보다 높으면 사람 라벨이 도움 → pseudo 기반 자산 |
| + PSEUDO | + Gemma 자가 라벨 고확신 양성 | 기반 자산보다 낮으면 자동으로 버림 |

그 다음에만 dev gold 사례를 제출용 static few-shot 자산에 추가하고 `submit.zip`을 만듭니다(같은 dev를 다시 채점하지 않음).

- `a100` 프로필은 평가 서버와 같은 설정(원본 모델·int8·16384)이라 dev 점수를 제출 성능 추정치로 볼 수 있습니다. `rtx3090` 프로필(AWQ 4bit·12288)은 상대 비교용입니다.
- 런타임이 끊기면 1번 셀 `RESUME_WORK`에 이전 WORK 경로를 넣고 처음부터 다시 실행하세요. 끝난 추론은 재사용되고, 끊긴 추론은 `<출력>.partial.jsonl`에서 이어갑니다.
- 추론 시간은 A100에서 dev 200건 1회 수~십수 분, 3090에서 수십 분 수준입니다. 빠르게 흐름만 보려면 `DEV_LIMIT = 20`, `RUN_PSEUDO = False`, `EVAL_HUMAN_HOLDOUT = False`.

최종 산출물: `submit.zip`(제출용), `pipeline_experiment_results.zip`(단계별 dev 결과·로그·요약) — `WORK_BASE`에 `*_latest.zip`으로도 복사
